Does adding semantic content to each historical trajectory event produce a more useful learned history representation?

In [1]:
import pandas as pd
import numpy as np

train_events = pd.read_csv(
    "../data/processed/trajectory_events_train.csv"
)

test_events = pd.read_csv(
    "../data/processed/trajectory_events_test.csv"
)

targets = pd.read_csv(
    "../data/processed/trajectory_targets.csv"
)

train_targets = (
    targets[
        targets["split"] == "train"
    ]
    .reset_index(drop=True)
)

test_targets = (
    targets[
        targets["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("Train events:", train_events.shape)
print("Test events:", test_events.shape)

print("Train targets:", train_targets.shape)
print("Test targets:", test_targets.shape)

assert len(train_targets) == 1489
assert len(test_targets) == 287

Train events: (3792, 40)
Test events: (799, 40)
Train targets: (1489, 14)
Test targets: (287, 14)


In [2]:
# ============================================================
# 2. Inspect trajectory-event schema
# ============================================================

print("Train event columns:")
print(train_events.columns.tolist())

print("\nTarget columns:")
print(train_targets.columns.tolist())

print("\nTrain event roles:")
print(
    train_events["event_role"]
    .value_counts(dropna=False)
)

print("\nExample events:")
display(
    train_events[
        [
            "dataset",
            "group_id",
            "message_index",
            "event_role",
            "content",
        ]
    ].head(10)
)

print("\nHistory lengths:")
display(
    train_targets["history_event_count"]
    .describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

Train event columns:
['dataset', 'group_id', 'trajectory_index', 'message_index', 'event_role', 'current_role', 'content', 'primary_tool', 'char_length', 'word_count', 'has_content', 'has_error_signal', 'is_system', 'is_user', 'is_assistant', 'is_tool_call', 'is_tool_result', 'raw_step_label', 'raw_step_reason', 'context_text', 'previous_messages', 'previous_tool_calls', 'previous_tool_results', 'previous_user_messages', 'previous_assistant_messages', 'context_char_length', 'context_word_count', 'canonical_group', 'split', 'failure_family', 'family_label', 'is_taxonomy_target', 'event_position', 'trajectory_event_count', 'relative_event_position', 'previous_error_signals', 'previous_event_role', 'previous_event_tool', 'role_transition', 'tool_transition']

Target columns:
['dataset', 'group_id', 'canonical_group', 'split', 'trajectory_index', 'message_index', 'event_position', 'history_event_count', 'has_history', 'event_role', 'primary_tool', 'content', 'family_label', 'failure_family

,dataset,group_id,message_index,event_role,content
0,A,a_1,2,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Austral..."
1,A,a_1,4,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query"": ""Australian ci..."
2,A,a_1,6,ASSISTANT,[ASSISTANT]\nThe Australian city founded in 18...
3,A,a_104,2,ASSISTANT,[ASSISTANT]\n<think>We need birth dates for Er...
4,A,a_104,4,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query"": ""Marco Da Silv..."
5,A,a_104,6,ASSISTANT,[ASSISTANT]\n<think>Erika Jayne (Erika Girardi...
6,A,a_105,2,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""univers..."
7,A,a_105,4,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers..."
8,A,a_105,6,TOOL_CALL,"[TOOL_CALL]\n\nsearch({""query_list"": [""Univers..."
9,A,a_105,8,ASSISTANT,[ASSISTANT]\nThe university with its main camp...



History lengths:


count    1489.000000
mean       15.756884
std        18.148802
min         0.000000
50%         9.000000
90%        42.000000
95%        59.600000
99%        82.120000
max        93.000000
Name: history_event_count, dtype: float64

In [3]:
# ============================================================
# 3. Stable trajectory keys
# ============================================================

TRAJECTORY_KEY = [
    "dataset",
    "group_id",
]

TARGET_KEY = [
    "dataset",
    "group_id",
    "message_index",
]


for df_name, df in [
    ("train_events", train_events),
    ("test_events", test_events),
    ("train_targets", train_targets),
    ("test_targets", test_targets),
]:

    missing = [
        col
        for col in TARGET_KEY
        if col not in df.columns
    ]

    assert not missing, (
        f"{df_name} missing columns: {missing}"
    )


# Event keys must be unique.
assert (
    train_events.duplicated(TARGET_KEY).sum()
    == 0
)

assert (
    test_events.duplicated(TARGET_KEY).sum()
    == 0
)

print("✓ Keys validated")

✓ Keys validated


In [4]:
# ============================================================
# 4. Build target -> historical-event mapping
# ============================================================

def build_history_indices(
    events_df,
    targets_df,
):

    # Map trajectory -> ordered event rows.
    trajectory_events = {}

    for trajectory_key, group in events_df.groupby(
        TRAJECTORY_KEY,
        sort=False,
    ):

        group = group.sort_values(
            "message_index"
        )

        trajectory_events[trajectory_key] = (
            group[
                [
                    "message_index",
                    "event_role",
                    "content",
                ]
            ]
            .copy()
        )

    histories = []

    for target_idx, row in targets_df.iterrows():

        key = (
            row["dataset"],
            row["group_id"],
        )

        target_message_index = int(
            row["message_index"]
        )

        events = trajectory_events.get(key)

        if events is None:

            history = []

        else:

            history = (
                events.loc[
                    events["message_index"]
                    < target_message_index
                ]
                .index
                .tolist()
            )

        histories.append(history)

    return histories


train_history_indices = build_history_indices(
    train_events,
    train_targets,
)

test_history_indices = build_history_indices(
    test_events,
    test_targets,
)


train_lengths = np.array(
    [len(x) for x in train_history_indices]
)

test_lengths = np.array(
    [len(x) for x in test_history_indices]
)

print("Train history:")
print(pd.Series(train_lengths).describe())

print("\nTest history:")
print(pd.Series(test_lengths).describe())

print(
    "\nTrain zero-history:",
    (train_lengths == 0).sum(),
)

print(
    "Test zero-history:",
    (test_lengths == 0).sum(),
)

Train history:
count    1489.000000
mean       15.756884
std        18.148802
min         0.000000
25%         5.000000
50%         9.000000
75%        18.000000
max        93.000000
dtype: float64

Test history:
count    287.000000
mean       9.296167
std        7.883289
min        0.000000
25%        3.000000
50%        8.000000
75%       13.000000
max       36.000000
dtype: float64

Train zero-history: 56
Test zero-history: 18


In [5]:
# ============================================================
# 5. Verify history reconstruction
# ============================================================

expected_train_lengths = (
    train_targets[
        "history_event_count"
    ]
    .to_numpy()
)

expected_test_lengths = (
    test_targets[
        "history_event_count"
    ]
    .to_numpy()
)


train_match = (
    train_lengths
    == expected_train_lengths
)

test_match = (
    test_lengths
    == expected_test_lengths
)


print(
    "Train exact history-count match:",
    train_match.mean(),
)

print(
    "Test exact history-count match:",
    test_match.mean(),
)


if not train_match.all():

    bad = np.where(~train_match)[0][:10]

    display(
        pd.DataFrame({
            "target_row": bad,
            "constructed":
                train_lengths[bad],
            "expected":
                expected_train_lengths[bad],
        })
    )


if not test_match.all():

    bad = np.where(~test_match)[0][:10]

    display(
        pd.DataFrame({
            "target_row": bad,
            "constructed":
                test_lengths[bad],
            "expected":
                expected_test_lengths[bad],
        })
    )


assert train_match.all()
assert test_match.all()

print(
    "\n✓ Historical-event reconstruction "
    "exactly matches export"
)

Train exact history-count match: 1.0
Test exact history-count match: 1.0

✓ Historical-event reconstruction exactly matches export


In [6]:
# ============================================================
# 6. Construct text-aware event representation
# ============================================================

def make_event_text(row):

    role = str(
        row["event_role"]
    ).strip()

    content = (
        ""
        if pd.isna(row["content"])
        else str(row["content"])
    )

    if role == "TOOL_CALL":
        prefix = "[TOOL_CALL]"

    elif role == "ASSISTANT":
        prefix = "[ASSISTANT]"

    else:
        prefix = f"[{role}]"

    return (
        prefix
        + "\n"
        + content
    )


train_events["event_text"] = (
    train_events.apply(
        make_event_text,
        axis=1,
    )
)

test_events["event_text"] = (
    test_events.apply(
        make_event_text,
        axis=1,
    )
)


print(
    train_events[
        [
            "event_role",
            "event_text",
        ]
    ]
    .head()
    .to_string(index=False)
)

event_role                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  event_text
 TOOL_CALL                                                                                                                                                                                                                                                                                                                                       

In [7]:
# ============================================================
# 7. Text sanity checks
# ============================================================

assert (
    train_events["event_text"]
    .notna()
    .all()
)

assert (
    test_events["event_text"]
    .notna()
    .all()
)


print("Train events:", len(train_events))
print("Test events:", len(test_events))

print(
    "\nTrain event-text lengths:"
)

print(
    train_events[
        "event_text"
    ]
    .str.len()
    .describe()
)

print(
    "\nTest event-text lengths:"
)

print(
    test_events[
        "event_text"
    ]
    .str.len()
    .describe()
)

Train events: 3792
Test events: 799

Train event-text lengths:
count    3792.000000
mean      303.365506
std       393.255354
min        23.000000
25%        64.000000
50%       134.000000
75%       383.250000
max      4370.000000
Name: event_text, dtype: float64

Test event-text lengths:
count     799.000000
mean      301.107635
std       398.261600
min        23.000000
25%        73.000000
50%       146.000000
75%       370.000000
max      4094.000000
Name: event_text, dtype: float64


In [8]:
# ============================================================
# 8. Clean text-aware event representation
# ============================================================

def make_event_text(row):

    role = str(
        row["event_role"]
    ).strip()

    content = (
        ""
        if pd.isna(row["content"])
        else str(row["content"]).strip()
    )

    role_prefix = f"[{role}]"

    # The exported current_text often already begins
    # with [ASSISTANT] or [TOOL_CALL].
    #
    # Avoid:
    #
    # [TOOL_CALL]
    # [TOOL_CALL]
    # search(...)
    #
    if content.upper().startswith(
        role_prefix.upper()
    ):
        return content

    return (
        role_prefix
        + "\n"
        + content
    )


train_events["event_text"] = (
    train_events.apply(
        make_event_text,
        axis=1,
    )
)

test_events["event_text"] = (
    test_events.apply(
        make_event_text,
        axis=1,
    )
)

In [9]:
for text in (
    train_events[
        "event_text"
    ]
    .head(5)
):

    print("=" * 80)
    print(text[:1000])

[TOOL_CALL]

search({"query_list": ["Australian city founded in 1838 with boarding school named after London school", "Australian boarding school founded by Prime Minister named after London school"]})
[TOOL_CALL]

search({"query": "Australian city founded in 1838 with boarding school opened by Prime Minister named after Westminster School in London"})
[ASSISTANT]
The Australian city founded in 1838 that contains a boarding school opened by a Prime Minister of Australia and named after a school in London of the same name is Adelaide.

Reasoning:
- The search results indicate that Adelaide has a boarding school named Westminster School, which is named after Westminster School in London.
- The school was opened by Prime Minister Robert Menzies in 1961, which fits the requirement of being opened by a Prime Minister.
- Adelaide was founded in 1838, as confirmed by the historical context provided in the search results.

Thus, the city is Adelaide.

<answer>Adelaide</answer>
[ASSISTANT]
<thi

In [10]:
from sentence_transformers import (
    SentenceTransformer,
)

semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [11]:
# ============================================================
# 10. Semantic embeddings for individual history events
# ============================================================

train_event_embeddings = (
    semantic_model.encode(
        train_events[
            "event_text"
        ].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

test_event_embeddings = (
    semantic_model.encode(
        test_events[
            "event_text"
        ].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

print(
    "Train event embeddings:",
    train_event_embeddings.shape,
)

print(
    "Test event embeddings:",
    test_event_embeddings.shape,
)

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Train event embeddings: (3792, 384)
Test event embeddings: (799, 384)


In [12]:
assert (
    len(train_event_embeddings)
    == len(train_events)
)

assert (
    len(test_event_embeddings)
    == len(test_events)
)

assert (
    train_event_embeddings.shape[1]
    == 384
)

print(
    "✓ Historical semantic embeddings ready"
)

✓ Historical semantic embeddings ready


In [13]:
# ============================================================
# 11. Current-message semantic representations
# ============================================================

train_current_embeddings = (
    semantic_model.encode(
        train_targets[
            "content"
        ]
        .fillna("")
        .astype(str)
        .tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

test_current_embeddings = (
    semantic_model.encode(
        test_targets[
            "content"
        ]
        .fillna("")
        .astype(str)
        .tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

print(
    train_current_embeddings.shape,
    test_current_embeddings.shape,
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

(1489, 384) (287, 384)


In [14]:
# ============================================================
# 12. Mean pooled historical semantics
# ============================================================

EMBEDDING_DIM = (
    train_event_embeddings.shape[1]
)


def mean_pool_histories(
    history_indices,
    event_embeddings,
):

    pooled = np.zeros(
        (
            len(history_indices),
            event_embeddings.shape[1],
        ),
        dtype=np.float32,
    )

    for target_idx, indices in enumerate(
        history_indices
    ):

        if len(indices) == 0:
            continue

        pooled[target_idx] = (
            event_embeddings[
                indices
            ].mean(
                axis=0
            )
        )

    return pooled


X_history_mean_train = (
    mean_pool_histories(
        train_history_indices,
        train_event_embeddings,
    )
)

X_history_mean_test = (
    mean_pool_histories(
        test_history_indices,
        test_event_embeddings,
    )
)


print(
    "Mean history train:",
    X_history_mean_train.shape,
)

print(
    "Mean history test:",
    X_history_mean_test.shape,
)

Mean history train: (1489, 384)
Mean history test: (287, 384)


In [15]:
# ============================================================
# 13. Recency-weighted semantic history
# ============================================================

def recency_pool_histories(
    history_indices,
    event_embeddings,
    decay=0.15,
):

    pooled = np.zeros(
        (
            len(history_indices),
            event_embeddings.shape[1],
        ),
        dtype=np.float32,
    )

    for target_idx, indices in enumerate(
        history_indices
    ):

        n = len(indices)

        if n == 0:
            continue

        # Oldest -> newest
        #
        # newest event gets distance 0.
        distances = np.arange(
            n - 1,
            -1,
            -1,
            dtype=np.float32,
        )

        weights = np.exp(
            -decay * distances
        )

        weights = (
            weights
            / weights.sum()
        )

        pooled[target_idx] = (
            event_embeddings[indices]
            * weights[:, None]
        ).sum(axis=0)

    return pooled


X_history_recency_train = (
    recency_pool_histories(
        train_history_indices,
        train_event_embeddings,
        decay=0.15,
    )
)

X_history_recency_test = (
    recency_pool_histories(
        test_history_indices,
        test_event_embeddings,
        decay=0.15,
    )
)

In [16]:
# ============================================================
# 14. Previous-event semantic representation
# ============================================================

def last_event_histories(
    history_indices,
    event_embeddings,
):

    result = np.zeros(
        (
            len(history_indices),
            event_embeddings.shape[1],
        ),
        dtype=np.float32,
    )

    for target_idx, indices in enumerate(
        history_indices
    ):

        if not indices:
            continue

        result[target_idx] = (
            event_embeddings[
                indices[-1]
            ]
        )

    return result


X_history_last_train = (
    last_event_histories(
        train_history_indices,
        train_event_embeddings,
    )
)

X_history_last_test = (
    last_event_histories(
        test_history_indices,
        test_event_embeddings,
    )
)

In [17]:
y_train = (
    train_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

y_test = (
    test_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

print(
    np.bincount(y_train)
)

print(
    np.bincount(y_test)
)

[660 317 237 244  31]
[138  70  38  30  11]


In [18]:
from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

In [19]:
def train_probe(
    X_train,
    X_test,
):

    model = LogisticRegression(
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_train,
        y_train,
    )

    pred = model.predict(
        X_test
    )

    result = {
        "accuracy":
            accuracy_score(
                y_test,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_test,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_test,
                pred,
                average="weighted",
                zero_division=0,
            ),
    }

    return (
        model,
        pred,
        result,
    )

In [20]:
current_model, current_pred, current_result = (
    train_probe(
        train_current_embeddings,
        test_current_embeddings,
    )
)

current_result

{'accuracy': 0.5156794425087108,
 'balanced_accuracy': 0.4813912250983189,
 'macro_f1': 0.4902679078131794,
 'weighted_f1': 0.5214873265322301}

In [21]:
mean_model, mean_pred, mean_result = (
    train_probe(
        X_history_mean_train,
        X_history_mean_test,
    )
)

print(
    "Mean history:",
    mean_result,
)

Mean history: {'accuracy': 0.5052264808362369, 'balanced_accuracy': 0.3782073761478796, 'macro_f1': 0.38884865416305586, 'weighted_f1': 0.4895218240095299}


In [22]:
recency_model, recency_pred, recency_result = (
    train_probe(
        X_history_recency_train,
        X_history_recency_test,
    )
)

print(
    "Recency history:",
    recency_result,
)

Recency history: {'accuracy': 0.43205574912891986, 'balanced_accuracy': 0.3539881323863018, 'macro_f1': 0.3535665877900263, 'weighted_f1': 0.4260495633989297}


In [23]:
last_model, last_pred, last_result = (
    train_probe(
        X_history_last_train,
        X_history_last_test,
    )
)

print(
    "Last event:",
    last_result,
)

Last event: {'accuracy': 0.5331010452961672, 'balanced_accuracy': 0.44769175904187347, 'macro_f1': 0.4494440796085046, 'weighted_f1': 0.511202414437574}


In [24]:
X_current_mean_train = np.hstack([
    train_current_embeddings,
    X_history_mean_train,
])

X_current_mean_test = np.hstack([
    test_current_embeddings,
    X_history_mean_test,
])


current_mean_model, current_mean_pred, current_mean_result = (
    train_probe(
        X_current_mean_train,
        X_current_mean_test,
    )
)

In [25]:
X_current_recency_train = np.hstack([
    train_current_embeddings,
    X_history_recency_train,
])

X_current_recency_test = np.hstack([
    test_current_embeddings,
    X_history_recency_test,
])


(
    current_recency_model,
    current_recency_pred,
    current_recency_result,
) = train_probe(
    X_current_recency_train,
    X_current_recency_test,
)

In [26]:
X_current_last_train = np.hstack([
    train_current_embeddings,
    X_history_last_train,
])

X_current_last_test = np.hstack([
    test_current_embeddings,
    X_history_last_test,
])


current_last_model, current_last_pred, current_last_result = (
    train_probe(
        X_current_last_train,
        X_current_last_test,
    )
)

In [28]:
results = pd.DataFrame([
    {
        "model":
            "current_semantic",
        **current_result,
    },
    {
        "model":
            "history_mean_semantic",
        **mean_result,
    },
    {
        "model":
            "history_recency_semantic",
        **recency_result,
    },
    {
        "model":
            "history_last_event",
        **last_result,
    },
    {
        "model":
            "current_plus_mean_history",
        **current_mean_result,
    },
    {
        "model":
            "current_plus_recency_history",
        **current_recency_result,
    },
    {
        "model":
            "current_plus_last_event",
        **current_last_result,
    },
])

results

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,current_semantic,0.515679,0.481391,0.490268,0.521487
1,history_mean_semantic,0.505226,0.378207,0.388849,0.489522
2,history_recency_semantic,0.432056,0.353988,0.353567,0.426050
3,history_last_event,0.533101,0.447692,0.449444,0.511202
4,current_plus_mean_history,0.505226,0.469015,0.475753,0.508890
5,current_plus_recency_history,0.512195,0.477131,0.481508,0.517207
6,current_plus_last_event,0.498258,0.468927,0.472012,0.503610


In [29]:
# ============================================================
# 21. Text-aware trajectory sequence dataset
# ============================================================

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


MAX_HISTORY = 96

# Current dataset max is 93, so nothing should be truncated yet.
print("Max train history:", max(map(len, train_history_indices)))
print("Max test history:", max(map(len, test_history_indices)))


class TextTrajectoryDataset(Dataset):

    def __init__(
        self,
        targets_df,
        history_indices,
        event_embeddings,
        current_embeddings,
    ):
        self.targets = (
            targets_df
            .reset_index(drop=True)
        )

        self.history_indices = history_indices

        self.event_embeddings = np.asarray(
            event_embeddings,
            dtype=np.float32,
        )

        self.current_embeddings = np.asarray(
            current_embeddings,
            dtype=np.float32,
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):

        indices = self.history_indices[idx]

        # Most recent MAX_HISTORY events.
        indices = indices[-MAX_HISTORY:]

        if len(indices) == 0:

            # One zero event so GRU always receives >= 1 step.
            history = np.zeros(
                (1, self.event_embeddings.shape[1]),
                dtype=np.float32,
            )

        else:

            history = self.event_embeddings[
                indices
            ]

        return {
            "history":
                torch.tensor(
                    history,
                    dtype=torch.float32,
                ),

            "current":
                torch.tensor(
                    self.current_embeddings[idx],
                    dtype=torch.float32,
                ),

            "label":
                torch.tensor(
                    int(
                        self.targets.iloc[idx][
                            "family_label"
                        ]
                    ),
                    dtype=torch.long,
                ),
        }

Max train history: 93
Max test history: 36


In [30]:
# ============================================================
# 22. Sequence padding
# ============================================================

def text_trajectory_collate(batch):

    lengths = torch.tensor(
        [
            len(item["history"])
            for item in batch
        ],
        dtype=torch.long,
    )

    history = pad_sequence(
        [
            item["history"]
            for item in batch
        ],
        batch_first=True,
        padding_value=0.0,
    )

    current = torch.stack(
        [
            item["current"]
            for item in batch
        ]
    )

    labels = torch.stack(
        [
            item["label"]
            for item in batch
        ]
    )

    return {
        "history": history,
        "lengths": lengths,
        "current": current,
        "labels": labels,
    }

In [31]:
# ============================================================
# 23. Group-safe inner train / validation split
# ============================================================

from sklearn.model_selection import GroupShuffleSplit


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

inner_train_idx, val_idx = next(
    splitter.split(
        train_targets,
        train_targets["family_label"],
        groups=train_targets["canonical_group"],
    )
)


inner_train_targets = (
    train_targets
    .iloc[inner_train_idx]
    .reset_index(drop=True)
)

val_targets = (
    train_targets
    .iloc[val_idx]
    .reset_index(drop=True)
)


inner_train_histories = [
    train_history_indices[i]
    for i in inner_train_idx
]

val_histories = [
    train_history_indices[i]
    for i in val_idx
]


inner_current_embeddings = (
    train_current_embeddings[
        inner_train_idx
    ]
)

val_current_embeddings = (
    train_current_embeddings[
        val_idx
    ]
)


assert not (
    set(inner_train_targets["canonical_group"])
    &
    set(val_targets["canonical_group"])
)

print("Inner train:", len(inner_train_targets))
print("Validation:", len(val_targets))
print("Group overlap: 0")

Inner train: 1185
Validation: 304
Group overlap: 0


In [32]:
# ============================================================
# 24. DataLoaders
# ============================================================

text_train_dataset = TextTrajectoryDataset(
    inner_train_targets,
    inner_train_histories,
    train_event_embeddings,
    inner_current_embeddings,
)

text_val_dataset = TextTrajectoryDataset(
    val_targets,
    val_histories,
    train_event_embeddings,
    val_current_embeddings,
)

text_test_dataset = TextTrajectoryDataset(
    test_targets,
    test_history_indices,
    test_event_embeddings,
    test_current_embeddings,
)


BATCH_SIZE = 32

text_train_loader = DataLoader(
    text_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=text_trajectory_collate,
)

text_val_loader = DataLoader(
    text_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=text_trajectory_collate,
)

text_test_loader = DataLoader(
    text_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=text_trajectory_collate,
)

print(
    len(text_train_dataset),
    len(text_val_dataset),
    len(text_test_dataset),
)

1185 304 287


In [33]:
# ============================================================
# 25. Text-aware trajectory GRU
# ============================================================

EVENT_EMBED_DIM = train_event_embeddings.shape[1]

TEXT_TRAJECTORY_DIM = 128


class TextAwareTrajectoryEncoder(nn.Module):

    def __init__(
        self,
        input_dim=384,
        hidden_dim=128,
        dropout=0.20,
    ):
        super().__init__()

        # Compress MiniLM event embeddings before GRU.
        self.input_projection = nn.Sequential(
            nn.Linear(
                input_dim,
                192,
            ),
            nn.ReLU(),
            nn.Dropout(0.10),
        )

        self.gru = nn.GRU(
            input_size=192,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        history,
        lengths,
    ):

        x = self.input_projection(
            history
        )

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, hidden = self.gru(
            packed
        )

        h = hidden[-1]

        return self.dropout(h)

In [34]:
# ============================================================
# 26. Text-aware history-only classifier
# ============================================================

NUM_CLASSES = 5


class TextTrajectoryOnlyModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = (
            TextAwareTrajectoryEncoder(
                input_dim=EVENT_EMBED_DIM,
                hidden_dim=TEXT_TRAJECTORY_DIM,
            )
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                TEXT_TRAJECTORY_DIM,
                64,
            ),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(
                64,
                NUM_CLASSES,
            ),
        )

    def forward(self, batch):

        trajectory = self.encoder(
            batch["history"],
            batch["lengths"],
        )

        return self.classifier(
            trajectory
        )

In [35]:
# ============================================================
# 27. Current semantic + text-aware trajectory
# ============================================================

CURRENT_DIM = (
    train_current_embeddings.shape[1]
)


class CurrentPlusTextTrajectoryModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = (
            TextAwareTrajectoryEncoder(
                input_dim=EVENT_EMBED_DIM,
                hidden_dim=TEXT_TRAJECTORY_DIM,
            )
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                CURRENT_DIM
                + TEXT_TRAJECTORY_DIM,
                256,
            ),

            nn.ReLU(),

            nn.Dropout(0.30),

            nn.Linear(
                256,
                64,
            ),

            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Linear(
                64,
                NUM_CLASSES,
            ),
        )

    def forward(self, batch):

        trajectory = self.encoder(
            batch["history"],
            batch["lengths"],
        )

        fused = torch.cat(
            [
                batch["current"],
                trajectory,
            ],
            dim=1,
        )

        return self.classifier(
            fused
        )

In [36]:
# ============================================================
# 28. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


In [37]:
# ============================================================
# 29. Evaluation helper
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)


def move_text_batch(
    batch,
    device,
):

    return {
        key: value.to(device)
        for key, value
        in batch.items()
    }


@torch.no_grad()
def evaluate_text_model(
    model,
    loader,
):

    model.eval()

    y_true = []
    y_pred = []

    for batch in loader:

        batch = move_text_batch(
            batch,
            device,
        )

        logits = model(
            batch
        )

        pred = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        labels = (
            batch["labels"]
            .cpu()
            .numpy()
        )

        y_true.extend(labels)
        y_pred.extend(pred)

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,
    }

In [38]:
# ============================================================
# 30. Training
# ============================================================

import copy


def train_text_model(
    model,
    train_loader,
    val_loader,
    epochs=30,
    lr=5e-4,
    patience=6,
):

    model = model.to(
        device
    )

    criterion = (
        nn.CrossEntropyLoss()
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    best_state = None
    best_macro_f1 = -np.inf

    patience_counter = 0

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        losses = []

        for batch in train_loader:

            batch = move_text_batch(
                batch,
                device,
            )

            optimizer.zero_grad()

            logits = model(
                batch
            )

            loss = criterion(
                logits,
                batch["labels"],
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            optimizer.step()

            losses.append(
                loss.item()
            )

        val_result = (
            evaluate_text_model(
                model,
                val_loader,
            )
        )

        row = {
            "epoch":
                epoch,

            "train_loss":
                np.mean(losses),

            "val_accuracy":
                val_result[
                    "accuracy"
                ],

            "val_balanced_accuracy":
                val_result[
                    "balanced_accuracy"
                ],

            "val_macro_f1":
                val_result[
                    "macro_f1"
                ],

            "val_weighted_f1":
                val_result[
                    "weighted_f1"
                ],
        }

        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"loss={row['train_loss']:.4f} | "
            f"val_macro_f1="
            f"{row['val_macro_f1']:.4f}"
        )

        if (
            row["val_macro_f1"]
            > best_macro_f1
        ):

            best_macro_f1 = (
                row["val_macro_f1"]
            )

            best_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            patience_counter
            >= patience
        ):

            print(
                "Early stopping."
            )

            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )

In [39]:
# ============================================================
# 31. Train text-aware trajectory-only model
# ============================================================

torch.manual_seed(42)
np.random.seed(42)

text_trajectory_model = (
    TextTrajectoryOnlyModel()
)

(
    text_trajectory_model,
    text_trajectory_history,
) = train_text_model(
    text_trajectory_model,
    text_train_loader,
    text_val_loader,
)

Epoch 01 | loss=1.4633 | val_macro_f1=0.1236
Epoch 02 | loss=1.3266 | val_macro_f1=0.1233
Epoch 03 | loss=1.2519 | val_macro_f1=0.1466
Epoch 04 | loss=1.1894 | val_macro_f1=0.2007
Epoch 05 | loss=1.1488 | val_macro_f1=0.2101
Epoch 06 | loss=1.1203 | val_macro_f1=0.2050
Epoch 07 | loss=1.1045 | val_macro_f1=0.2530
Epoch 08 | loss=1.0922 | val_macro_f1=0.3308
Epoch 09 | loss=1.0747 | val_macro_f1=0.3714
Epoch 10 | loss=1.0432 | val_macro_f1=0.4322
Epoch 11 | loss=1.0878 | val_macro_f1=0.4576
Epoch 12 | loss=1.0276 | val_macro_f1=0.4305
Epoch 13 | loss=0.9513 | val_macro_f1=0.4405
Epoch 14 | loss=0.9414 | val_macro_f1=0.4436
Epoch 15 | loss=1.0096 | val_macro_f1=0.4437
Epoch 16 | loss=0.9451 | val_macro_f1=0.4785
Epoch 17 | loss=0.9394 | val_macro_f1=0.4791
Epoch 18 | loss=0.8795 | val_macro_f1=0.4686
Epoch 19 | loss=0.8727 | val_macro_f1=0.4401
Epoch 20 | loss=0.8425 | val_macro_f1=0.4098
Epoch 21 | loss=0.8340 | val_macro_f1=0.4499
Epoch 22 | loss=0.8349 | val_macro_f1=0.4689
Epoch 23 |

In [40]:
# ============================================================
# 32. Evaluate text-aware trajectory model
# ============================================================

text_trajectory_result = (
    evaluate_text_model(
        text_trajectory_model,
        text_test_loader,
    )
)

print({
    k: v
    for k, v
    in text_trajectory_result.items()
    if k not in {
        "y_true",
        "y_pred",
    }
})

{'accuracy': 0.3554006968641115, 'balanced_accuracy': 0.38722101696930067, 'macro_f1': 0.3819446735962162, 'weighted_f1': 0.3606321280173699}


In [41]:
sequence_comparison = pd.DataFrame([
    {
        "model":
            "current_semantic",
        **current_result,
    },
    {
        "model":
            "history_mean",
        **mean_result,
    },
    {
        "model":
            "history_recency",
        **recency_result,
    },
    {
        "model":
            "history_last_event",
        **last_result,
    },
    {
        "model":
            "text_aware_trajectory_gru",
        **{
            k:
                text_trajectory_result[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },
])

display(
    sequence_comparison
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,current_semantic,0.5157,0.4814,0.4903,0.5215
3,history_last_event,0.5331,0.4477,0.4494,0.5112
1,history_mean,0.5052,0.3782,0.3888,0.4895
4,text_aware_trajectory_gru,0.3554,0.3872,0.3819,0.3606
2,history_recency,0.4321,0.3540,0.3536,0.4260


In [42]:
CurrentPlusTextTrajectoryModel()

CurrentPlusTextTrajectoryModel(
  (encoder): TextAwareTrajectoryEncoder(
    (input_projection): Sequential(
      (0): Linear(in_features=384, out_features=192, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.1, inplace=False)
    )
    (gru): GRU(192, 128, batch_first=True)
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=5, bias=True)
  )
)

In [43]:
# ============================================================
# 33. Last-K event semantic representations
# ============================================================

def last_k_pool_histories(
    history_indices,
    event_embeddings,
    k,
):
    pooled = np.zeros(
        (
            len(history_indices),
            event_embeddings.shape[1],
        ),
        dtype=np.float32,
    )

    for target_idx, indices in enumerate(
        history_indices
    ):

        if len(indices) == 0:
            continue

        recent = indices[-k:]

        pooled[target_idx] = (
            event_embeddings[
                recent
            ]
            .mean(axis=0)
        )

    return pooled

In [44]:
history_windows = [
    1,
    2,
    3,
    5,
    10,
]

window_representations = {}

for k in history_windows:

    X_train_k = last_k_pool_histories(
        train_history_indices,
        train_event_embeddings,
        k=k,
    )

    X_test_k = last_k_pool_histories(
        test_history_indices,
        test_event_embeddings,
        k=k,
    )

    window_representations[k] = (
        X_train_k,
        X_test_k,
    )

    print(
        f"k={k}",
        X_train_k.shape,
        X_test_k.shape,
    )

k=1 (1489, 384) (287, 384)
k=2 (1489, 384) (287, 384)
k=3 (1489, 384) (287, 384)
k=5 (1489, 384) (287, 384)
k=10 (1489, 384) (287, 384)


In [45]:
window_results = []

window_predictions = {}

for k in history_windows:

    X_train_k, X_test_k = (
        window_representations[k]
    )

    model_k, pred_k, result_k = (
        train_probe(
            X_train_k,
            X_test_k,
        )
    )

    window_predictions[k] = pred_k

    window_results.append({
        "history_window": k,
        **result_k,
    })


window_results_df = pd.DataFrame(
    window_results
)

display(
    window_results_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,history_window,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,1,0.5331,0.4477,0.4494,0.5112
2,3,0.4843,0.4398,0.4462,0.4799
1,2,0.4913,0.4436,0.4457,0.4865
4,10,0.4495,0.3494,0.3587,0.4422
3,5,0.3763,0.3224,0.3202,0.3763


In [46]:
fusion_window_results = []

fusion_window_predictions = {}

for k in history_windows:

    X_train_k, X_test_k = (
        window_representations[k]
    )

    X_train_fused = np.hstack([
        train_current_embeddings,
        X_train_k,
    ])

    X_test_fused = np.hstack([
        test_current_embeddings,
        X_test_k,
    ])

    model_k, pred_k, result_k = (
        train_probe(
            X_train_fused,
            X_test_fused,
        )
    )

    fusion_window_predictions[k] = (
        pred_k
    )

    fusion_window_results.append({
        "history_window": k,
        **result_k,
    })


fusion_window_results_df = (
    pd.DataFrame(
        fusion_window_results
    )
)

display(
    fusion_window_results_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,history_window,accuracy,balanced_accuracy,macro_f1,weighted_f1
2,3,0.5122,0.4876,0.4888,0.5196
4,10,0.5122,0.4824,0.4857,0.5175
0,1,0.4983,0.4689,0.4720,0.5036
1,2,0.5017,0.4676,0.4718,0.5096
3,5,0.5017,0.4647,0.4718,0.5081


In [47]:
baseline_macro_f1 = (
    current_result["macro_f1"]
)

comparison = (
    fusion_window_results_df
    .copy()
)

comparison[
    "delta_macro_f1_vs_current"
] = (
    comparison["macro_f1"]
    - baseline_macro_f1
)

display(
    comparison
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,history_window,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_macro_f1_vs_current
2,3,0.5122,0.4876,0.4888,0.5196,-0.0014
4,10,0.5122,0.4824,0.4857,0.5175,-0.0046
0,1,0.4983,0.4689,0.4720,0.5036,-0.0183
1,2,0.5017,0.4676,0.4718,0.5096,-0.0185
3,5,0.5017,0.4647,0.4718,0.5081,-0.0185


In [48]:
# ============================================================
# 34. Fixed-position recent-history embeddings
# ============================================================

def get_last_k_event_embeddings(
    history_indices,
    event_embeddings,
    k,
):
    """
    Returns shape:
        (n_targets, k, embedding_dim)

    Order:
        oldest -> newest within the selected last-k window

    Missing positions are zero-padded on the left.
    """

    n_targets = len(history_indices)
    embedding_dim = event_embeddings.shape[1]

    output = np.zeros(
        (
            n_targets,
            k,
            embedding_dim,
        ),
        dtype=np.float32,
    )

    for target_idx, indices in enumerate(
        history_indices
    ):
        if len(indices) == 0:
            continue

        recent = indices[-k:]

        values = event_embeddings[
            recent
        ]

        # Left pad so newest event is always at position -1.
        output[
            target_idx,
            -len(recent):,
            :
        ] = values

    return output

In [49]:
recent_history = {}

for k in [1, 2, 3]:

    H_train_k = (
        get_last_k_event_embeddings(
            train_history_indices,
            train_event_embeddings,
            k=k,
        )
    )

    H_test_k = (
        get_last_k_event_embeddings(
            test_history_indices,
            test_event_embeddings,
            k=k,
        )
    )

    recent_history[k] = (
        H_train_k,
        H_test_k,
    )

    print(
        f"k={k}",
        H_train_k.shape,
        H_test_k.shape,
    )

k=1 (1489, 1, 384) (287, 1, 384)
k=2 (1489, 2, 384) (287, 2, 384)
k=3 (1489, 3, 384) (287, 3, 384)


In [50]:
# ============================================================
# 35. Current <-> history relational representation
# ============================================================

def build_relational_features(
    current_embeddings,
    history_embeddings,
):
    """
    current_embeddings:
        (N, D)

    history_embeddings:
        (N, K, D)

    Output:
        [current,
         history_i,
         abs(current-history_i),
         current*history_i
         for each history position]
    """

    current_embeddings = np.asarray(
        current_embeddings,
        dtype=np.float32,
    )

    history_embeddings = np.asarray(
        history_embeddings,
        dtype=np.float32,
    )

    features = [
        current_embeddings
    ]

    k = history_embeddings.shape[1]

    for position in range(k):

        h = history_embeddings[
            :,
            position,
            :
        ]

        abs_diff = np.abs(
            current_embeddings - h
        )

        interaction = (
            current_embeddings * h
        )

        features.extend([
            h,
            abs_diff,
            interaction,
        ])

    return np.hstack(
        features
    )

In [51]:
relational_features = {}

for k in [1, 2, 3]:

    H_train_k, H_test_k = (
        recent_history[k]
    )

    X_train_rel = (
        build_relational_features(
            train_current_embeddings,
            H_train_k,
        )
    )

    X_test_rel = (
        build_relational_features(
            test_current_embeddings,
            H_test_k,
        )
    )

    relational_features[k] = (
        X_train_rel,
        X_test_rel,
    )

    print(
        f"k={k}",
        X_train_rel.shape,
        X_test_rel.shape,
    )

k=1 (1489, 1536) (287, 1536)
k=2 (1489, 2688) (287, 2688)
k=3 (1489, 3840) (287, 3840)


In [52]:
# ============================================================
# 36. Relational LR probe
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def train_relational_probe(
    X_train,
    X_test,
    C=0.1,
):
    model = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=C,
            max_iter=5000,
            random_state=42,
        ),
    )

    model.fit(
        X_train,
        y_train,
    )

    pred = model.predict(
        X_test
    )

    result = {
        "accuracy":
            accuracy_score(
                y_test,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_test,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_test,
                pred,
                average="weighted",
                zero_division=0,
            ),
    }

    return (
        model,
        pred,
        result,
    )

In [53]:
# ============================================================
# 37. Evaluate relational history windows
# ============================================================

relational_results = []

relational_predictions = {}

relational_models = {}

for k in [1, 2, 3]:

    X_train_rel, X_test_rel = (
        relational_features[k]
    )

    model_k, pred_k, result_k = (
        train_relational_probe(
            X_train_rel,
            X_test_rel,
            C=0.1,
        )
    )

    relational_models[k] = (
        model_k
    )

    relational_predictions[k] = (
        pred_k
    )

    relational_results.append({
        "history_window": k,
        **result_k,
    })


relational_results_df = (
    pd.DataFrame(
        relational_results
    )
)

display(
    relational_results_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,history_window,accuracy,balanced_accuracy,macro_f1,weighted_f1
1,2,0.5122,0.4534,0.4678,0.5187
2,3,0.4564,0.4186,0.4349,0.4600
0,1,0.4530,0.3856,0.4020,0.4593


In [54]:
# ============================================================
# 38. Main comparison
# ============================================================

comparison_rows = [
    {
        "model":
            "current_semantic",

        **current_result,
    },
    {
        "model":
            "current_plus_last1_mean",

        **fusion_window_results_df.loc[
            fusion_window_results_df[
                "history_window"
            ] == 1
        ]
        .iloc[0]
        .drop(
            labels=[
                "history_window"
            ]
        )
        .to_dict(),
    },
    {
        "model":
            "current_plus_last3_mean",

        **fusion_window_results_df.loc[
            fusion_window_results_df[
                "history_window"
            ] == 3
        ]
        .iloc[0]
        .drop(
            labels=[
                "history_window"
            ]
        )
        .to_dict(),
    },
]


for _, row in (
    relational_results_df
    .iterrows()
):

    comparison_rows.append({
        "model":
            f"relational_last_{int(row['history_window'])}",

        "accuracy":
            row["accuracy"],

        "balanced_accuracy":
            row[
                "balanced_accuracy"
            ],

        "macro_f1":
            row["macro_f1"],

        "weighted_f1":
            row[
                "weighted_f1"
            ],
    })


relational_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)

In [55]:
baseline = (
    relational_comparison
    .set_index("model")
    .loc[
        "current_semantic"
    ]
)

for metric in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:

    relational_comparison[
        f"delta_{metric}_vs_current"
    ] = (
        relational_comparison[
            metric
        ]
        - baseline[
            metric
        ]
    )


display(
    relational_comparison
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_current,delta_balanced_accuracy_vs_current,delta_macro_f1_vs_current,delta_weighted_f1_vs_current
0,current_semantic,0.5157,0.4814,0.4903,0.5215,0.0000,0.0000,0.0000,0.0000
2,current_plus_last3_mean,0.5122,0.4876,0.4888,0.5196,-0.0035,0.0062,-0.0014,-0.0019
1,current_plus_last1_mean,0.4983,0.4689,0.4720,0.5036,-0.0174,-0.0125,-0.0183,-0.0179
4,relational_last_2,0.5122,0.4534,0.4678,0.5187,-0.0035,-0.0280,-0.0225,-0.0028
5,relational_last_3,0.4564,0.4186,0.4349,0.4600,-0.0592,-0.0628,-0.0554,-0.0615
3,relational_last_1,0.4530,0.3856,0.4020,0.4593,-0.0627,-0.0958,-0.0882,-0.0622


In [60]:
# ============================================================
# 39. Best relational model class report
# ============================================================

from sklearn.metrics import classification_report


best_k = int(
    relational_results_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0][
        "history_window"
    ]
)

best_relational_pred = (
    relational_predictions[
        best_k
    ]
)

print(
    "Best relational window:",
    best_k,
)

print(
    classification_report(
        y_test,
        best_relational_pred,
        target_names=[
            "workflow_error",
            "constraint_error",
            "tool_use_error",
            "grounding_state_error",
            "reasoning_value_error",
        ],
        digits=4,
        zero_division=0,
    )
)

Best relational window: 2
                       precision    recall  f1-score   support

       workflow_error     0.6535    0.6014    0.6264       138
     constraint_error     0.4932    0.5143    0.5035        70
       tool_use_error     0.2564    0.2632    0.2597        38
grounding_state_error     0.3095    0.4333    0.3611        30
reasoning_value_error     0.8333    0.4545    0.5882        11

             accuracy                         0.5122       287
            macro avg     0.5092    0.4534    0.4678       287
         weighted avg     0.5328    0.5122    0.5187       287



In [57]:
# ============================================================
# 40. Current-history semantic similarity
# ============================================================

def cosine_similarity_rows(
    A,
    B,
):
    numerator = (
        A * B
    ).sum(axis=1)

    denominator = (
        np.linalg.norm(
            A,
            axis=1,
        )
        *
        np.linalg.norm(
            B,
            axis=1,
        )
        + 1e-8
    )

    return (
        numerator
        / denominator
    )

In [58]:
H_train_3, H_test_3 = (
    recent_history[3]
)

similarity_test = pd.DataFrame({
    "sim_t_minus_3":
        cosine_similarity_rows(
            test_current_embeddings,
            H_test_3[:, 0, :],
        ),

    "sim_t_minus_2":
        cosine_similarity_rows(
            test_current_embeddings,
            H_test_3[:, 1, :],
        ),

    "sim_t_minus_1":
        cosine_similarity_rows(
            test_current_embeddings,
            H_test_3[:, 2, :],
        ),

    "failure_family":
        test_targets[
            "failure_family"
        ].to_numpy(),
})

In [59]:
display(
    similarity_test
    .groupby(
        "failure_family"
    )[
        [
            "sim_t_minus_3",
            "sim_t_minus_2",
            "sim_t_minus_1",
        ]
    ]
    .agg([
        "mean",
        "median",
        "std",
    ])
    .round(4)
)

sim_t_minus_3                 sim_t_minus_2          \
                               mean  median     std          mean  median   
failure_family                                                              
constraint_error             0.4452  0.4543  0.2863        0.4320  0.3804   
grounding_state_error        0.2399  0.1463  0.2764        0.3555  0.3757   
reasoning_value_error        0.3793  0.3472  0.2251        0.4788  0.4143   
tool_use_error               0.2922  0.1500  0.3080        0.3640  0.3713   
workflow_error               0.4996  0.5018  0.3023        0.5280  0.5246   

                              sim_t_minus_1                  
                          std          mean  median     std  
failure_family                                               
constraint_error       0.2569        0.4864  0.4659  0.2323  
grounding_state_error  0.2898        0.4102  0.4203  0.2299  
reasoning_value_error  0.2009        0.5589  0.4547  0.1850  
tool_use_error         0.3372        0.4279  0.4418  0.3311  
workflow_error         0.3084        0.5644  0.5440  0.2688

In [61]:
# ============================================================
# 41. Compact semantic transition features
# ============================================================

def build_transition_features(
    current_embeddings,
    history_embeddings,
):
    """
    Compact transition representation.

    history_embeddings:
        (N, K, D), left-padded with zeros.

    Features per historical position:
        - cosine similarity(current, history)
        - L1 distance
        - L2 distance
        - dot product
        - history-present indicator

    Plus sequential history-history transition features.
    """

    current = np.asarray(
        current_embeddings,
        dtype=np.float32,
    )

    history = np.asarray(
        history_embeddings,
        dtype=np.float32,
    )

    N, K, D = history.shape

    feature_columns = []
    feature_names = []

    def cosine_rows(a, b):
        numerator = np.sum(
            a * b,
            axis=1,
        )

        denominator = (
            np.linalg.norm(a, axis=1)
            * np.linalg.norm(b, axis=1)
            + 1e-8
        )

        return numerator / denominator

    # --------------------------------------------------------
    # Current <-> each historical event
    # --------------------------------------------------------

    for pos in range(K):

        h = history[:, pos, :]

        present = (
            np.linalg.norm(
                h,
                axis=1,
            ) > 1e-8
        ).astype(np.float32)

        cosine = cosine_rows(
            current,
            h,
        )

        l1 = np.mean(
            np.abs(current - h),
            axis=1,
        )

        l2 = np.linalg.norm(
            current - h,
            axis=1,
        )

        dot = np.sum(
            current * h,
            axis=1,
        )

        # Remove meaningless values for padded events.
        cosine *= present
        l1 *= present
        l2 *= present
        dot *= present

        relative_pos = K - pos

        feature_columns.extend([
            cosine,
            l1,
            l2,
            dot,
            present,
        ])

        feature_names.extend([
            f"current_tminus{relative_pos}_cosine",
            f"current_tminus{relative_pos}_l1",
            f"current_tminus{relative_pos}_l2",
            f"current_tminus{relative_pos}_dot",
            f"tminus{relative_pos}_present",
        ])

    # --------------------------------------------------------
    # Historical transition:
    #
    # h(t-3) -> h(t-2)
    # h(t-2) -> h(t-1)
    # --------------------------------------------------------

    for pos in range(1, K):

        previous = history[:, pos - 1, :]
        newer = history[:, pos, :]

        previous_present = (
            np.linalg.norm(
                previous,
                axis=1,
            ) > 1e-8
        )

        newer_present = (
            np.linalg.norm(
                newer,
                axis=1,
            ) > 1e-8
        )

        pair_present = (
            previous_present
            & newer_present
        ).astype(np.float32)

        cosine = cosine_rows(
            previous,
            newer,
        ) * pair_present

        l2 = np.linalg.norm(
            newer - previous,
            axis=1,
        ) * pair_present

        feature_columns.extend([
            cosine,
            l2,
        ])

        feature_names.extend([
            f"history_transition_{pos}_cosine",
            f"history_transition_{pos}_l2",
        ])

    X = np.column_stack(
        feature_columns
    ).astype(np.float32)

    return X, feature_names

In [62]:
# ============================================================
# 42. Build compact transition features
# ============================================================

H_train_3, H_test_3 = recent_history[3]

X_transition_train, transition_feature_names = (
    build_transition_features(
        train_current_embeddings,
        H_train_3,
    )
)

X_transition_test, _ = (
    build_transition_features(
        test_current_embeddings,
        H_test_3,
    )
)

print(
    "Transition train:",
    X_transition_train.shape,
)

print(
    "Transition test:",
    X_transition_test.shape,
)

print("\nFeatures:")

for name in transition_feature_names:
    print(" ", name)

Transition train: (1489, 19)
Transition test: (287, 19)

Features:
  current_tminus3_cosine
  current_tminus3_l1
  current_tminus3_l2
  current_tminus3_dot
  tminus3_present
  current_tminus2_cosine
  current_tminus2_l1
  current_tminus2_l2
  current_tminus2_dot
  tminus2_present
  current_tminus1_cosine
  current_tminus1_l1
  current_tminus1_l2
  current_tminus1_dot
  tminus1_present
  history_transition_1_cosine
  history_transition_1_l2
  history_transition_2_cosine
  history_transition_2_l2


In [63]:
# ============================================================
# 43. Transition-only probe
# ============================================================

transition_only_model = make_pipeline(
    StandardScaler(),

    LogisticRegression(
        C=0.1,
        max_iter=5000,
        random_state=42,
    ),
)

transition_only_model.fit(
    X_transition_train,
    y_train,
)

transition_only_pred = (
    transition_only_model.predict(
        X_transition_test
    )
)

transition_only_result = {
    "accuracy":
        accuracy_score(
            y_test,
            transition_only_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            transition_only_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            transition_only_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            transition_only_pred,
            average="weighted",
            zero_division=0,
        ),
}

print(
    transition_only_result
)

{'accuracy': 0.4529616724738676, 'balanced_accuracy': 0.20547455595510516, 'macro_f1': 0.1680879727432445, 'weighted_f1': 0.3449780655268577}


In [64]:
# ============================================================
# 44. Current semantic + compact transition features
# ============================================================

X_transition_fused_train = np.hstack([
    train_current_embeddings,
    X_transition_train,
])

X_transition_fused_test = np.hstack([
    test_current_embeddings,
    X_transition_test,
])

transition_fused_model = make_pipeline(
    StandardScaler(),

    LogisticRegression(
        C=0.1,
        max_iter=5000,
        random_state=42,
    ),
)

transition_fused_model.fit(
    X_transition_fused_train,
    y_train,
)

transition_fused_pred = (
    transition_fused_model.predict(
        X_transition_fused_test
    )
)

transition_fused_result = {
    "accuracy":
        accuracy_score(
            y_test,
            transition_fused_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            transition_fused_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            transition_fused_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            transition_fused_pred,
            average="weighted",
            zero_division=0,
        ),
}

print(
    transition_fused_result
)

{'accuracy': 0.43205574912891986, 'balanced_accuracy': 0.42020406748095535, 'macro_f1': 0.42050060042717413, 'weighted_f1': 0.44513963808536217}


In [65]:
# ============================================================
# 45. Regularization sweep
# ============================================================

transition_sweep = []

transition_sweep_predictions = {}

for C in [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
]:

    model = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=C,
            max_iter=5000,
            random_state=42,
        ),
    )

    model.fit(
        X_transition_fused_train,
        y_train,
    )

    pred = model.predict(
        X_transition_fused_test
    )

    result = {
        "C": C,

        "accuracy":
            accuracy_score(
                y_test,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_test,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_test,
                pred,
                average="weighted",
                zero_division=0,
            ),
    }

    transition_sweep.append(
        result
    )

    transition_sweep_predictions[C] = pred


transition_sweep_df = pd.DataFrame(
    transition_sweep
)

display(
    transition_sweep_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,C,accuracy,balanced_accuracy,macro_f1,weighted_f1
1,0.003,0.5226,0.4877,0.4949,0.5255
0,0.001,0.5366,0.4514,0.4753,0.5368
2,0.010,0.4843,0.4595,0.4661,0.4915
3,0.030,0.4704,0.4366,0.4431,0.4808
6,1.000,0.4251,0.4389,0.4315,0.4412
5,0.300,0.4286,0.4232,0.4255,0.4409
4,0.100,0.4321,0.4202,0.4205,0.4451


In [66]:
# ============================================================
# 46. Transition features by true failure family
# ============================================================

transition_analysis = pd.DataFrame(
    X_transition_test,
    columns=transition_feature_names,
)

transition_analysis[
    "failure_family"
] = (
    test_targets[
        "failure_family"
    ].to_numpy()
)

display(
    transition_analysis
    .groupby(
        "failure_family"
    )
    .mean()
    .round(4)
)

,current_tminus3_cosine,current_tminus3_l1,current_tminus3_l2,current_tminus3_dot,tminus3_present,current_tminus2_cosine,current_tminus2_l1,current_tminus2_l2,current_tminus2_dot,tminus2_present,current_tminus1_cosine,current_tminus1_l1,current_tminus1_l2,current_tminus1_dot,tminus1_present,history_transition_1_cosine,history_transition_1_l2,history_transition_2_cosine,history_transition_2_l2
failure_family,,,,,,,,,,,,,,,,,,,
constraint_error,0.4452,0.0338,0.8330,0.4452,0.8857,0.4320,0.0389,0.9565,0.4320,0.9571,0.4864,0.0379,0.9336,0.4864,0.9714,0.4607,0.8048,0.4655,0.9192
grounding_state_error,0.2399,0.0258,0.6393,0.2399,0.6000,0.3555,0.0332,0.8196,0.3555,0.8000,0.4102,0.0376,0.9221,0.4102,0.9000,0.2442,0.6422,0.4389,0.7149
reasoning_value_error,0.3793,0.0381,0.9530,0.3793,0.9091,0.4788,0.0398,0.9890,0.4788,1.0000,0.5589,0.0371,0.9158,0.5589,1.0000,0.7255,0.5006,0.6661,0.7613
tool_use_error,0.2922,0.0237,0.5853,0.2922,0.6053,0.3640,0.0257,0.6296,0.3640,0.6842,0.4279,0.0287,0.7058,0.4279,0.7895,0.2616,0.6149,0.3492,0.6403
workflow_error,0.4996,0.0310,0.7655,0.4996,0.8768,0.5280,0.0318,0.7850,0.5280,0.9203,0.5644,0.0332,0.8195,0.5644,0.9638,0.5291,0.7289,0.5678,0.7412


In [67]:
# ============================================================
# 47. Recover group IDs
# ============================================================

print(train_targets.columns.tolist())

group_col_candidates = [
    "canonical_group",
    "group_id",
]

group_col = next(
    (
        col
        for col in group_col_candidates
        if col in train_targets.columns
    ),
    None,
)

if group_col is None:
    raise ValueError(
        "Could not find group column in train_targets"
    )

groups = (
    train_targets[group_col]
    .astype(str)
    .to_numpy()
)

print("Using group column:", group_col)
print("Training rows:", len(groups))
print("Unique groups:", len(np.unique(groups)))

assert len(groups) == len(y_train)

['dataset', 'group_id', 'canonical_group', 'split', 'trajectory_index', 'message_index', 'event_position', 'history_event_count', 'has_history', 'event_role', 'primary_tool', 'content', 'family_label', 'failure_family']
Using group column: canonical_group
Training rows: 1489
Unique groups: 335


In [68]:
# ============================================================
# 48. Remove redundant transition features
# ============================================================

keep_transition_features = [
    name
    for name in transition_feature_names
    if not name.endswith("_dot")
]

transition_keep_idx = [
    transition_feature_names.index(name)
    for name in keep_transition_features
]

X_transition_train_reduced = (
    X_transition_train[
        :,
        transition_keep_idx,
    ]
)

X_transition_test_reduced = (
    X_transition_test[
        :,
        transition_keep_idx,
    ]
)

print(
    "Original:",
    X_transition_train.shape,
)

print(
    "Reduced:",
    X_transition_train_reduced.shape,
)

print("\nRetained features:")

for feature in keep_transition_features:
    print(" ", feature)

Original: (1489, 19)
Reduced: (1489, 16)

Retained features:
  current_tminus3_cosine
  current_tminus3_l1
  current_tminus3_l2
  tminus3_present
  current_tminus2_cosine
  current_tminus2_l1
  current_tminus2_l2
  tminus2_present
  current_tminus1_cosine
  current_tminus1_l1
  current_tminus1_l2
  tminus1_present
  history_transition_1_cosine
  history_transition_1_l2
  history_transition_2_cosine
  history_transition_2_l2


In [69]:
X_compact_train = np.hstack([
    train_current_embeddings,
    X_transition_train_reduced,
])

X_compact_test = np.hstack([
    test_current_embeddings,
    X_transition_test_reduced,
])

print(
    X_compact_train.shape,
    X_compact_test.shape,
)

(1489, 400) (287, 400)


In [70]:
# ============================================================
# 49. Group-safe CV hyperparameter selection
# ============================================================

from sklearn.model_selection import (
    StratifiedGroupKFold,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


C_VALUES = [
    0.0003,
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
]


cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [71]:
# ============================================================
# 50. CV compact-transition fusion
# ============================================================

cv_rows = []

for C in C_VALUES:

    fold_metrics = []

    for fold, (
        train_idx,
        val_idx,
    ) in enumerate(
        cv.split(
            X_compact_train,
            y_train,
            groups=groups,
        ),
        start=1,
    ):

        model = make_pipeline(
            StandardScaler(),

            LogisticRegression(
                C=C,
                max_iter=5000,
                random_state=42,
            ),
        )

        model.fit(
            X_compact_train[train_idx],
            y_train[train_idx],
        )

        pred = model.predict(
            X_compact_train[val_idx]
        )

        fold_metrics.append({
            "accuracy":
                accuracy_score(
                    y_train[val_idx],
                    pred,
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_train[val_idx],
                    pred,
                ),

            "macro_f1":
                f1_score(
                    y_train[val_idx],
                    pred,
                    average="macro",
                    zero_division=0,
                ),

            "weighted_f1":
                f1_score(
                    y_train[val_idx],
                    pred,
                    average="weighted",
                    zero_division=0,
                ),
        })

    fold_df = pd.DataFrame(
        fold_metrics
    )

    cv_rows.append({
        "C": C,

        "cv_accuracy":
            fold_df[
                "accuracy"
            ].mean(),

        "cv_balanced_accuracy":
            fold_df[
                "balanced_accuracy"
            ].mean(),

        "cv_macro_f1":
            fold_df[
                "macro_f1"
            ].mean(),

        "cv_macro_f1_std":
            fold_df[
                "macro_f1"
            ].std(),

        "cv_weighted_f1":
            fold_df[
                "weighted_f1"
            ].mean(),
    })


compact_cv_results = pd.DataFrame(
    cv_rows
)

display(
    compact_cv_results
    .sort_values(
        "cv_macro_f1",
        ascending=False,
    )
    .round(4)
)

,C,cv_accuracy,cv_balanced_accuracy,cv_macro_f1,cv_macro_f1_std,cv_weighted_f1
3,0.0100,0.5353,0.4820,0.4969,0.0733,0.5299
4,0.0300,0.5252,0.4849,0.4947,0.0625,0.5196
5,0.1000,0.5191,0.4838,0.4894,0.0577,0.5154
2,0.0030,0.5393,0.4544,0.4707,0.0279,0.5279
6,0.3000,0.5137,0.4662,0.4672,0.0393,0.5102
1,0.0010,0.5258,0.3742,0.3815,0.0314,0.5049
0,0.0003,0.5064,0.3268,0.3333,0.0461,0.4655


In [72]:
# ============================================================
# 51. Select hyperparameter from CV only
# ============================================================

best_cv_row = (
    compact_cv_results
    .sort_values(
        "cv_macro_f1",
        ascending=False,
    )
    .iloc[0]
)

best_C = float(
    best_cv_row["C"]
)

print("Selected C:", best_C)

print(
    best_cv_row.round(4)
)

Selected C: 0.01
C                       0.0100
cv_accuracy             0.5353
cv_balanced_accuracy    0.4820
cv_macro_f1             0.4969
cv_macro_f1_std         0.0733
cv_weighted_f1          0.5299
Name: 3, dtype: float64


In [73]:
# ============================================================
# 52. Final compact-transition model
# ============================================================

final_transition_model = make_pipeline(
    StandardScaler(),

    LogisticRegression(
        C=best_C,
        max_iter=5000,
        random_state=42,
    ),
)

final_transition_model.fit(
    X_compact_train,
    y_train,
)

final_transition_pred = (
    final_transition_model.predict(
        X_compact_test
    )
)

final_transition_result = {
    "accuracy":
        accuracy_score(
            y_test,
            final_transition_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            final_transition_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            final_transition_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            final_transition_pred,
            average="weighted",
            zero_division=0,
        ),
}

print(
    final_transition_result
)

print()

print(
    classification_report(
        y_test,
        final_transition_pred,
        target_names=[
            "workflow_error",
            "constraint_error",
            "tool_use_error",
            "grounding_state_error",
            "reasoning_value_error",
        ],
        digits=4,
        zero_division=0,
    )
)

{'accuracy': 0.4843205574912892, 'balanced_accuracy': 0.4594995393622396, 'macro_f1': 0.46607369620073785, 'weighted_f1': 0.4915041450830039}

                       precision    recall  f1-score   support

       workflow_error     0.5935    0.5290    0.5594       138
     constraint_error     0.4865    0.5143    0.5000        70
       tool_use_error     0.3421    0.3421    0.3421        38
grounding_state_error     0.2500    0.3667    0.2973        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.4843       287
            macro avg     0.4844    0.4595    0.4661       287
         weighted avg     0.5042    0.4843    0.4915       287



In [74]:
# ============================================================
# 53. Group-safe CV semantic baseline
# ============================================================

semantic_cv_rows = []

for C in C_VALUES:

    fold_macro_f1 = []

    for train_idx, val_idx in cv.split(
        train_current_embeddings,
        y_train,
        groups=groups,
    ):

        model = make_pipeline(
            StandardScaler(),

            LogisticRegression(
                C=C,
                max_iter=5000,
                random_state=42,
            ),
        )

        model.fit(
            train_current_embeddings[
                train_idx
            ],
            y_train[train_idx],
        )

        pred = model.predict(
            train_current_embeddings[
                val_idx
            ]
        )

        fold_macro_f1.append(
            f1_score(
                y_train[val_idx],
                pred,
                average="macro",
                zero_division=0,
            )
        )

    semantic_cv_rows.append({
        "C": C,
        "cv_macro_f1":
            np.mean(fold_macro_f1),
        "cv_macro_f1_std":
            np.std(
                fold_macro_f1,
                ddof=1,
            ),
    })


semantic_cv_results = pd.DataFrame(
    semantic_cv_rows
)

display(
    semantic_cv_results
    .sort_values(
        "cv_macro_f1",
        ascending=False,
    )
    .round(4)
)

,C,cv_macro_f1,cv_macro_f1_std
4,0.0300,0.4967,0.0604
3,0.0100,0.4856,0.0485
5,0.1000,0.4795,0.0602
6,0.3000,0.4752,0.0326
2,0.0030,0.4701,0.0368
1,0.0010,0.3842,0.0436
0,0.0003,0.3336,0.0403


In [75]:
best_semantic_C = float(
    semantic_cv_results
    .sort_values(
        "cv_macro_f1",
        ascending=False,
    )
    .iloc[0]["C"]
)

print(
    "Selected semantic C:",
    best_semantic_C,
)


final_semantic_model = make_pipeline(
    StandardScaler(),

    LogisticRegression(
        C=best_semantic_C,
        max_iter=5000,
        random_state=42,
    ),
)

final_semantic_model.fit(
    train_current_embeddings,
    y_train,
)

final_semantic_pred = (
    final_semantic_model.predict(
        test_current_embeddings
    )
)

final_semantic_result = {
    "accuracy":
        accuracy_score(
            y_test,
            final_semantic_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            final_semantic_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            final_semantic_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            final_semantic_pred,
            average="weighted",
            zero_division=0,
        ),
}

print(
    final_semantic_result
)

Selected semantic C: 0.03
{'accuracy': 0.445993031358885, 'balanced_accuracy': 0.47024597065787, 'macro_f1': 0.461352711394966, 'weighted_f1': 0.4581954161623312}


In [76]:
# ============================================================
# 54. Fair final comparison
# ============================================================

fair_comparison = pd.DataFrame([
    {
        "model":
            "semantic_cv_selected",
        "C":
            best_semantic_C,
        **final_semantic_result,
    },
    {
        "model":
            "semantic_plus_compact_transition",
        "C":
            best_C,
        **final_transition_result,
    },
])

baseline = fair_comparison.iloc[0]

for metric in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:

    fair_comparison[
        f"delta_{metric}"
    ] = (
        fair_comparison[metric]
        - baseline[metric]
    )

display(
    fair_comparison.round(4)
)

,model,C,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy,delta_balanced_accuracy,delta_macro_f1,delta_weighted_f1
0,semantic_cv_selected,0.03,0.4460,0.4702,0.4614,0.4582,0.0000,0.0000,0.0000,0.0000
1,semantic_plus_compact_transition,0.01,0.4843,0.4595,0.4661,0.4915,0.0383,-0.0107,0.0047,0.0333


In [77]:
# ============================================================
# 55. Semantic vs transition rescue analysis
# ============================================================

semantic_correct = (
    final_semantic_pred == y_test
)

transition_correct = (
    final_transition_pred == y_test
)

rescued = (
    (~semantic_correct)
    & transition_correct
)

broken = (
    semantic_correct
    & (~transition_correct)
)

both_correct = (
    semantic_correct
    & transition_correct
)

both_wrong = (
    (~semantic_correct)
    & (~transition_correct)
)


print("Test examples:", len(y_test))

print(
    "Semantic correct:",
    semantic_correct.sum(),
)

print(
    "Transition correct:",
    transition_correct.sum(),
)

print(
    "Rescues:",
    rescued.sum(),
)

print(
    "Breaks:",
    broken.sum(),
)

print(
    "Net rescues:",
    rescued.sum() - broken.sum(),
)

print(
    "Both correct:",
    both_correct.sum(),
)

print(
    "Both wrong:",
    both_wrong.sum(),
)

Test examples: 287
Semantic correct: 128
Transition correct: 139
Rescues: 31
Breaks: 20
Net rescues: 11
Both correct: 108
Both wrong: 128


In [78]:
# ============================================================
# 56. Rescue / break behavior by true class
# ============================================================

LABEL_NAMES = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]


rescue_analysis = pd.DataFrame({
    "true_label": y_test,
    "semantic_pred": final_semantic_pred,
    "transition_pred": final_transition_pred,
    "semantic_correct": semantic_correct,
    "transition_correct": transition_correct,
    "rescued": rescued,
    "broken": broken,
})


rescue_analysis["failure_family"] = [
    LABEL_NAMES[int(y)]
    for y in y_test
]


class_rescue = (
    rescue_analysis
    .groupby("failure_family")
    .agg(
        count=("true_label", "size"),

        semantic_correct=(
            "semantic_correct",
            "sum",
        ),

        transition_correct=(
            "transition_correct",
            "sum",
        ),

        rescues=(
            "rescued",
            "sum",
        ),

        breaks=(
            "broken",
            "sum",
        ),
    )
)


class_rescue["net_rescues"] = (
    class_rescue["rescues"]
    - class_rescue["breaks"]
)


class_rescue["rescue_rate"] = (
    class_rescue["rescues"]
    / class_rescue["count"]
)


class_rescue["break_rate"] = (
    class_rescue["breaks"]
    / class_rescue["count"]
)


display(
    class_rescue
    .sort_values(
        "net_rescues",
        ascending=False,
    )
    .round(4)
)

,count,semantic_correct,transition_correct,rescues,breaks,net_rescues,rescue_rate,break_rate
failure_family,,,,,,,,
workflow_error,138,57,73,24,8,16,0.1739,0.0580
constraint_error,70,34,36,5,3,2,0.0714,0.0429
reasoning_value_error,11,6,6,0,0,0,0.0000,0.0000
grounding_state_error,30,13,11,2,4,-2,0.0667,0.1333
tool_use_error,38,18,13,0,5,-5,0.0000,0.1316


In [79]:
# ============================================================
# 57. Transition characteristics of rescues
# ============================================================

transition_diagnostics = (
    pd.DataFrame(
        X_transition_test_reduced,
        columns=keep_transition_features,
    )
)

transition_diagnostics[
    "outcome"
] = np.select(
    [
        rescued,
        broken,
        both_correct,
    ],
    [
        "rescue",
        "break",
        "both_correct",
    ],
    default="both_wrong",
)


transition_diagnostics[
    "failure_family"
] = [
    LABEL_NAMES[int(y)]
    for y in y_test
]


display(
    transition_diagnostics
    .groupby("outcome")[
        keep_transition_features
    ]
    .mean()
    .T
    .round(4)
)

outcome,both_correct,both_wrong,break,rescue
current_tminus3_cosine,0.4454,0.3734,0.3842,0.6125
current_tminus3_l1,0.0346,0.0278,0.0312,0.0264
current_tminus3_l2,0.8551,0.6883,0.7656,0.6478
tminus3_present,0.8981,0.7266,0.8000,0.9032
current_tminus2_cosine,0.4725,0.4248,0.3502,0.6601
current_tminus2_l1,0.0365,0.0317,0.0390,0.0237
current_tminus2_l2,0.8995,0.7823,0.9642,0.5844
tminus2_present,0.9537,0.8281,0.9000,0.9032
current_tminus1_cosine,0.5485,0.4411,0.4203,0.7273
current_tminus1_l1,0.0356,0.0356,0.0354,0.0242


In [80]:
# ============================================================
# 58. Cosine transition analysis
# ============================================================

cosine_cols = [
    col
    for col in keep_transition_features
    if "cosine" in col
]


cosine_summary = (
    transition_diagnostics
    .groupby("outcome")[
        cosine_cols
    ]
    .agg([
        "mean",
        "median",
        "std",
    ])
)


display(
    cosine_summary.round(4)
)

current_tminus3_cosine                 current_tminus2_cosine  \
                               mean  median     std                   mean   
outcome                                                                      
both_correct                 0.4454  0.4359  0.2888                 0.4725   
both_wrong                   0.3734  0.4233  0.2957                 0.4248   
break                        0.3842  0.3830  0.3132                 0.3502   
rescue                       0.6125  0.6249  0.3425                 0.6601   

                             current_tminus1_cosine                  \
              median     std                   mean  median     std   
outcome                                                               
both_correct  0.4427  0.2814                 0.5485  0.5478  0.2432   
both_wrong    0.4551  0.2956                 0.4411  0.4551  0.2523   
break         0.3420  0.2842                 0.4203  0.4460  0.2992   
rescue        0.7770  0.3291                 0.7273  0.8755  0.2629   

             history_transition_1_cosine                  \
                                    mean  median     std   
outcome                                                    
both_correct                      0.5115  0.5440  0.3074   
both_wrong                        0.3730  0.3769  0.3161   
break                             0.4201  0.4674  0.3267   
rescue                            0.6167  0.8034  0.3511   

             history_transition_2_cosine                  
                                    mean  median     std  
outcome                                                   
both_correct                      0.5088  0.5267  0.2824  
both_wrong                        0.4707  0.5183  0.3233  
break                             0.4718  0.4580  0.3302  
rescue                            0.6476  0.7443  0.3141

In [81]:
for col in cosine_cols:

    print("\n", "=" * 70)
    print(col)
    print("=" * 70)

    display(
        transition_diagnostics
        .groupby("outcome")[col]
        .describe()[
            [
                "count",
                "mean",
                "std",
                "25%",
                "50%",
                "75%",
            ]
        ]
        .round(4)
    )


current_tminus3_cosine


,count,mean,std,25%,50%,75%
outcome,,,,,,
both_correct,108.0,0.4454,0.2888,0.2153,0.4359,0.6399
both_wrong,128.0,0.3734,0.2957,0.0000,0.4233,0.6223
break,20.0,0.3842,0.3132,0.0984,0.3830,0.4949
rescue,31.0,0.6125,0.3425,0.4334,0.6249,0.9021



current_tminus2_cosine


,count,mean,std,25%,50%,75%
outcome,,,,,,
both_correct,108.0,0.4725,0.2814,0.2531,0.4427,0.6706
both_wrong,128.0,0.4248,0.2956,0.1955,0.4551,0.6602
break,20.0,0.3502,0.2842,0.1702,0.3420,0.4734
rescue,31.0,0.6601,0.3291,0.4414,0.7770,0.9262



current_tminus1_cosine


,count,mean,std,25%,50%,75%
outcome,,,,,,
both_correct,108.0,0.5485,0.2432,0.3498,0.5478,0.7422
both_wrong,128.0,0.4411,0.2523,0.2982,0.4551,0.6307
break,20.0,0.4203,0.2992,0.2367,0.4460,0.5012
rescue,31.0,0.7273,0.2629,0.4809,0.8755,0.9386



history_transition_1_cosine


,count,mean,std,25%,50%,75%
outcome,,,,,,
both_correct,108.0,0.5115,0.3074,0.2679,0.5440,0.7640
both_wrong,128.0,0.3730,0.3161,0.0000,0.3769,0.6005
break,20.0,0.4201,0.3267,0.0858,0.4674,0.5773
rescue,31.0,0.6167,0.3511,0.4855,0.8034,0.8807



history_transition_2_cosine


,count,mean,std,25%,50%,75%
outcome,,,,,,
both_correct,108.0,0.5088,0.2824,0.3074,0.5267,0.6930
both_wrong,128.0,0.4707,0.3233,0.1815,0.5183,0.7268
break,20.0,0.4718,0.3302,0.2597,0.4580,0.7501
rescue,31.0,0.6476,0.3141,0.4648,0.7443,0.8887


In [82]:
# ============================================================
# 59. Semantic -> transition prediction changes
# ============================================================

prediction_changes = pd.DataFrame({
    "semantic": [
        LABEL_NAMES[int(x)]
        for x in final_semantic_pred
    ],

    "transition": [
        LABEL_NAMES[int(x)]
        for x in final_transition_pred
    ],

    "truth": [
        LABEL_NAMES[int(x)]
        for x in y_test
    ],
})


change_matrix = pd.crosstab(
    prediction_changes["semantic"],
    prediction_changes["transition"],
)


display(change_matrix)

transition,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
semantic,,,,,
constraint_error,52,2,0,1,3
grounding_state_error,8,37,0,0,4
reasoning_value_error,0,0,8,0,0
tool_use_error,3,1,0,36,25
workflow_error,11,4,0,1,91


In [83]:
disagreements = prediction_changes[
    prediction_changes["semantic"]
    != prediction_changes["transition"]
].copy()


print(
    "Prediction disagreements:",
    len(disagreements),
)

print(
    "Disagreement rate:",
    len(disagreements) / len(y_test),
)


display(
    disagreements
    .groupby(
        [
            "semantic",
            "transition",
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
    .head(20)
)

Prediction disagreements: 63
Disagreement rate: 0.21951219512195122


semantic               transition           
tool_use_error         workflow_error           25
workflow_error         constraint_error         11
grounding_state_error  constraint_error          8
                       workflow_error            4
workflow_error         grounding_state_error     4
constraint_error       workflow_error            3
tool_use_error         constraint_error          3
constraint_error       grounding_state_error     2
                       tool_use_error            1
tool_use_error         grounding_state_error     1
workflow_error         tool_use_error            1
dtype: int64

In [84]:
# ============================================================
# 60. Oracle semantic/transition gate
# ============================================================

oracle_correct = (
    semantic_correct
    | transition_correct
)

oracle_accuracy = (
    oracle_correct.mean()
)


print(
    "Semantic accuracy:",
    semantic_correct.mean(),
)

print(
    "Transition accuracy:",
    transition_correct.mean(),
)

print(
    "Oracle accuracy:",
    oracle_accuracy,
)

print(
    "Oracle gain over semantic:",
    oracle_accuracy
    - semantic_correct.mean(),
)

Semantic accuracy: 0.445993031358885
Transition accuracy: 0.4843205574912892
Oracle accuracy: 0.554006968641115
Oracle gain over semantic: 0.10801393728222997


In [86]:
# ============================================================
# 60A. Reconstruct feature matrices for OOF experiment
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. Semantic features
# ------------------------------------------------------------
#
# Earlier in this notebook you encoded the current target text.
# Find the names of those embedding arrays first.
#
# This prints likely embedding variables currently in memory.

[
    (name, getattr(value, "shape", None))
    for name, value in globals().items()
    if (
        isinstance(value, np.ndarray)
        and getattr(value, "ndim", 0) == 2
        and (
            "emb" in name.lower()
            or "semantic" in name.lower()
        )
    )
]

[('train_event_embeddings', (3792, 384)),
 ('test_event_embeddings', (799, 384)),
 ('train_current_embeddings', (1489, 384)),
 ('test_current_embeddings', (287, 384)),
 ('inner_current_embeddings', (1185, 384)),
 ('val_current_embeddings', (304, 384))]

In [87]:
X_sem_train = train_current_embeddings
X_sem_test = test_current_embeddings

In [88]:
# ============================================================
# 60A. Define semantic and transition expert matrices
# ============================================================

X_sem_train = np.asarray(
    train_current_embeddings,
    dtype=np.float32,
)

X_sem_test = np.asarray(
    test_current_embeddings,
    dtype=np.float32,
)


print(
    "Semantic train:",
    X_sem_train.shape,
)

print(
    "Semantic test:",
    X_sem_test.shape,
)

Semantic train: (1489, 384)
Semantic test: (287, 384)


In [89]:
candidate_transition_arrays = [
    (name, value.shape)
    for name, value in globals().items()
    if (
        isinstance(value, np.ndarray)
        and value.ndim == 2
        and value.shape[0] in {1489, 287}
        and "transition" in name.lower()
    )
]

candidate_transition_arrays

[('X_transition_train', (1489, 19)),
 ('X_transition_test', (287, 19)),
 ('X_transition_fused_train', (1489, 403)),
 ('X_transition_fused_test', (287, 403)),
 ('X_transition_train_reduced', (1489, 16)),
 ('X_transition_test_reduced', (287, 16))]

In [90]:
# ============================================================
# 60B. Build transition expert
# ============================================================

T_train = np.asarray(
    X_transition_train_reduced,
    dtype=np.float32,
)

T_test = np.asarray(
    X_transition_test_reduced,
    dtype=np.float32,
)

print(
    "Transition-only features:",
    T_train.shape,
    T_test.shape,
)


X_trans_train = np.hstack([
    X_sem_train,
    T_train,
])

X_trans_test = np.hstack([
    X_sem_test,
    T_test,
])


print(
    "Transition expert train:",
    X_trans_train.shape,
)

print(
    "Transition expert test:",
    X_trans_test.shape,
)

Transition-only features: (1489, 16) (287, 16)
Transition expert train: (1489, 400)
Transition expert test: (287, 400)


In [91]:
T_train = X_transition_train
T_test = X_transition_test

In [92]:
# ============================================================
# 60C. Labels and groups
# ============================================================

y_train_int = (
    train_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

y_test_int = (
    test_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)


group_col = (
    "canonical_group"
    if "canonical_group"
    in train_targets.columns
    else "group_id"
)

groups = (
    train_targets[
        group_col
    ]
    .astype(str)
    .to_numpy()
)


print(
    "Group column:",
    group_col,
)

print(
    "Train rows:",
    len(y_train_int),
)

print(
    "Unique groups:",
    len(np.unique(groups)),
)


assert X_sem_train.shape[0] == 1489
assert X_trans_train.shape[0] == 1489
assert len(y_train_int) == 1489
assert len(groups) == 1489

print(
    "✓ Expert matrices aligned"
)

Group column: canonical_group
Train rows: 1489
Unique groups: 335
✓ Expert matrices aligned


In [93]:
# ============================================================
# 61. Group-safe OOF predictions for both experts
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


n_classes = 5

oof_sem_prob = np.zeros(
    (
        len(y_train_int),
        n_classes,
    ),
    dtype=np.float32,
)

oof_trans_prob = np.zeros(
    (
        len(y_train_int),
        n_classes,
    ),
    dtype=np.float32,
)


cv_oof = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


for fold, (
    tr_idx,
    va_idx,
) in enumerate(
    cv_oof.split(
        X_sem_train,
        y_train_int,
        groups=groups,
    ),
    start=1,
):

    # --------------------------------------------------------
    # Semantic expert
    # --------------------------------------------------------

    semantic_model_fold = (
        make_pipeline(
            StandardScaler(),

            LogisticRegression(
                C=0.03,
                max_iter=5000,
                random_state=42,
            ),
        )
    )

    semantic_model_fold.fit(
        X_sem_train[tr_idx],
        y_train_int[tr_idx],
    )

    oof_sem_prob[va_idx] = (
        semantic_model_fold.predict_proba(
            X_sem_train[va_idx]
        )
    )


    # --------------------------------------------------------
    # Transition expert
    # --------------------------------------------------------

    transition_model_fold = (
        make_pipeline(
            StandardScaler(),

            LogisticRegression(
                C=0.01,
                max_iter=5000,
                random_state=42,
            ),
        )
    )

    transition_model_fold.fit(
        X_trans_train[tr_idx],
        y_train_int[tr_idx],
    )

    oof_trans_prob[va_idx] = (
        transition_model_fold.predict_proba(
            X_trans_train[va_idx]
        )
    )


    print(
        f"Fold {fold} complete"
    )


oof_sem_pred = (
    oof_sem_prob.argmax(axis=1)
)

oof_trans_pred = (
    oof_trans_prob.argmax(axis=1)
)

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [94]:
# ============================================================
# 62. OOF oracle opportunity
# ============================================================

sem_correct_oof = (
    oof_sem_pred
    == y_train_int
)

trans_correct_oof = (
    oof_trans_pred
    == y_train_int
)

disagree_oof = (
    oof_sem_pred
    != oof_trans_pred
)

rescue_oof = (
    (~sem_correct_oof)
    & trans_correct_oof
)

break_oof = (
    sem_correct_oof
    & (~trans_correct_oof)
)


oracle_oof = (
    sem_correct_oof
    | trans_correct_oof
)


print(
    "OOF semantic accuracy:",
    sem_correct_oof.mean(),
)

print(
    "OOF transition accuracy:",
    trans_correct_oof.mean(),
)

print(
    "OOF disagreement rate:",
    disagree_oof.mean(),
)

print(
    "OOF rescues:",
    rescue_oof.sum(),
)

print(
    "OOF breaks:",
    break_oof.sum(),
)

print(
    "OOF net rescues:",
    rescue_oof.sum()
    - break_oof.sum(),
)

print(
    "OOF oracle accuracy:",
    oracle_oof.mean(),
)

print(
    "OOF oracle gain over semantic:",
    oracle_oof.mean()
    - sem_correct_oof.mean(),
)

OOF semantic accuracy: 0.5251846877098724
OOF transition accuracy: 0.5352585627938213
OOF disagreement rate: 0.1229012760241773
OOF rescues: 69
OOF breaks: 54
OOF net rescues: 15
OOF oracle accuracy: 0.5715245130960376
OOF oracle gain over semantic: 0.04633982538616521


In [95]:
# ============================================================
# 63. Build disagreement-router training set
# ============================================================

router_mask = (
    disagree_oof
    &
    (
        sem_correct_oof
        ^ trans_correct_oof
    )
)

router_target = (
    trans_correct_oof[
        router_mask
    ]
    .astype(int)
)

print(
    "Router training rows:",
    router_mask.sum(),
)

print(
    "Trust semantic:",
    (router_target == 0).sum(),
)

print(
    "Trust transition:",
    (router_target == 1).sum(),
)

print(
    "Transition share:",
    router_target.mean(),
)

Router training rows: 123
Trust semantic: 54
Trust transition: 69
Transition share: 0.5609756097560976


In [97]:
# ============================================================
# 64. Router features
# ============================================================

def probability_features(probabilities):
    sorted_prob = np.sort(
        probabilities,
        axis=1,
    )

    confidence = (
        sorted_prob[:, -1]
    )

    margin = (
        sorted_prob[:, -1]
        - sorted_prob[:, -2]
    )

    entropy = -np.sum(
        probabilities
        * np.log(
            probabilities + 1e-8
        ),
        axis=1,
    )

    return np.column_stack([
        confidence,
        margin,
        entropy,
    ])


sem_meta = probability_features(
    oof_sem_prob
)

trans_meta = probability_features(
    oof_trans_prob
)

In [98]:
expert_meta = np.column_stack([
    oof_sem_prob,
    oof_trans_prob,
    sem_meta,
    trans_meta,

    # Difference between expert probabilities
    oof_trans_prob
    - oof_sem_prob,
])

In [99]:
router_X_full = np.hstack([
    expert_meta,
    T_train,
])

router_X = (
    router_X_full[
        router_mask
    ]
)

router_y = router_target

print(
    "Router X:",
    router_X.shape,
)

print(
    "Router y:",
    router_y.shape,
)

Router X: (123, 40)
Router y: (123,)


In [100]:
router_groups = (
    groups[
        router_mask
    ]
)

print(
    "Router groups:",
    len(
        np.unique(
            router_groups
        )
    )
)

Router groups: 87


In [101]:
# ============================================================
# 65. Group-safe router CV
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)


ROUTER_C_VALUES = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
]


router_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

router_cv_rows = []


for C in ROUTER_C_VALUES:

    fold_rows = []

    for tr_idx, va_idx in router_cv.split(
        router_X,
        router_y,
        groups=router_groups,
    ):

        model = make_pipeline(
            StandardScaler(),

            LogisticRegression(
                C=C,
                max_iter=5000,
                random_state=42,
            ),
        )

        model.fit(
            router_X[tr_idx],
            router_y[tr_idx],
        )

        pred = model.predict(
            router_X[va_idx]
        )

        fold_rows.append({
            "accuracy":
                accuracy_score(
                    router_y[va_idx],
                    pred,
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    router_y[va_idx],
                    pred,
                ),

            "macro_f1":
                f1_score(
                    router_y[va_idx],
                    pred,
                    average="macro",
                    zero_division=0,
                ),
        })

    fold_df = pd.DataFrame(
        fold_rows
    )

    router_cv_rows.append({
        "C": C,
        "accuracy":
            fold_df[
                "accuracy"
            ].mean(),

        "balanced_accuracy":
            fold_df[
                "balanced_accuracy"
            ].mean(),

        "macro_f1":
            fold_df[
                "macro_f1"
            ].mean(),

        "macro_f1_std":
            fold_df[
                "macro_f1"
            ].std(),
    })


router_cv_results = pd.DataFrame(
    router_cv_rows
)

display(
    router_cv_results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,C,accuracy,balanced_accuracy,macro_f1,macro_f1_std
5,0.300,0.5847,0.5749,0.5706,0.1435
4,0.100,0.5690,0.5576,0.5554,0.1551
6,1.000,0.5447,0.5327,0.5283,0.1370
3,0.030,0.5543,0.5350,0.5281,0.1564
2,0.010,0.5290,0.4934,0.4610,0.0824
1,0.003,0.5444,0.4882,0.3871,0.0551
0,0.001,0.5617,0.5000,0.3595,0.0126


In [102]:
always_semantic_router_acc = (
    (router_y == 0)
    .mean()
)

always_transition_router_acc = (
    (router_y == 1)
    .mean()
)

print(
    "Always semantic router:",
    always_semantic_router_acc,
)

print(
    "Always transition router:",
    always_transition_router_acc,
)

Always semantic router: 0.43902439024390244
Always transition router: 0.5609756097560976


In [103]:
# ============================================================
# 66. Cross-fitted router predictions
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

router_oof_choice = np.full(
    len(y_train_int),
    -1,
    dtype=int,
)

# Only examples where experts disagree matter.
disagreement_indices = np.where(
    disagree_oof
)[0]

X_router_disagree = (
    router_X_full[
        disagreement_indices
    ]
)

y_router_disagree = (
    trans_correct_oof[
        disagreement_indices
    ]
    .astype(int)
)

groups_router_disagree = (
    groups[
        disagreement_indices
    ]
)

print(
    "Disagreement examples:",
    len(disagreement_indices),
)

print(
    "Transition correct:",
    y_router_disagree.sum(),
)

print(
    "Semantic correct:",
    (
        sem_correct_oof[
            disagreement_indices
        ]
    ).sum(),
)

print(
    "Neither correct:",
    (
        ~sem_correct_oof[
            disagreement_indices
        ]
        &
        ~trans_correct_oof[
            disagreement_indices
        ]
    ).sum(),
)

Disagreement examples: 183
Transition correct: 69
Semantic correct: 54
Neither correct: 60


In [104]:
# ============================================================
# 67. Cross-fit router
# ============================================================

informative_mask = (
    disagree_oof
    &
    (
        sem_correct_oof
        ^ trans_correct_oof
    )
)

informative_indices = np.where(
    informative_mask
)[0]

X_router_info = (
    router_X_full[
        informative_indices
    ]
)

y_router_info = (
    trans_correct_oof[
        informative_indices
    ]
    .astype(int)
)

groups_router_info = (
    groups[
        informative_indices
    ]
)


router_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

router_crossfit_pred = np.full(
    len(y_train_int),
    -1,
    dtype=int,
)


for fold, (tr, va) in enumerate(
    router_cv.split(
        X_router_info,
        y_router_info,
        groups=groups_router_info,
    ),
    start=1,
):

    model = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=0.3,
            max_iter=5000,
            random_state=42,
        ),
    )

    model.fit(
        X_router_info[tr],
        y_router_info[tr],
    )

    original_val_idx = (
        informative_indices[va]
    )

    router_crossfit_pred[
        original_val_idx
    ] = model.predict(
        X_router_info[va]
    )

    print(
        f"Fold {fold} complete"
    )

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [105]:
# ============================================================
# 68. End-to-end routed OOF predictions
# ============================================================

# Default = semantic expert
routed_oof_pred = (
    oof_sem_pred.copy()
)

routable = (
    router_crossfit_pred >= 0
)

# Switch to transition only where router says 1
switch_to_transition = (
    routable
    &
    (
        router_crossfit_pred == 1
    )
)

routed_oof_pred[
    switch_to_transition
] = (
    oof_trans_pred[
        switch_to_transition
    ]
)


print(
    "Routable examples:",
    routable.sum(),
)

print(
    "Switched to transition:",
    switch_to_transition.sum(),
)

Routable examples: 123
Switched to transition: 68


In [106]:
# ============================================================
# 69. Compare end-to-end OOF systems
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)


def evaluate_predictions(
    y_true,
    pred,
):

    return {
        "accuracy":
            accuracy_score(
                y_true,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                pred,
                average="weighted",
                zero_division=0,
            ),
    }


comparison = pd.DataFrame([
    {
        "model": "semantic",
        **evaluate_predictions(
            y_train_int,
            oof_sem_pred,
        ),
    },
    {
        "model": "transition",
        **evaluate_predictions(
            y_train_int,
            oof_trans_pred,
        ),
    },
    {
        "model": "learned_router",
        **evaluate_predictions(
            y_train_int,
            routed_oof_pred,
        ),
    },
])


display(
    comparison.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic,0.5252,0.4908,0.4974,0.5231
1,transition,0.5353,0.4839,0.4981,0.5323
2,learned_router,0.5373,0.4865,0.4973,0.5348


In [107]:
# ============================================================
# 70. Router rescue/break analysis
# ============================================================

semantic_correct = (
    oof_sem_pred
    == y_train_int
)

routed_correct = (
    routed_oof_pred
    == y_train_int
)

router_rescues = (
    (~semantic_correct)
    &
    routed_correct
)

router_breaks = (
    semantic_correct
    &
    (~routed_correct)
)


print(
    "Semantic correct:",
    semantic_correct.sum(),
)

print(
    "Routed correct:",
    routed_correct.sum(),
)

print(
    "Router rescues:",
    router_rescues.sum(),
)

print(
    "Router breaks:",
    router_breaks.sum(),
)

print(
    "Net rescues:",
    router_rescues.sum()
    - router_breaks.sum(),
)

print(
    "Accuracy delta:",
    routed_correct.mean()
    - semantic_correct.mean(),
)

Semantic correct: 782
Routed correct: 800
Router rescues: 43
Router breaks: 25
Net rescues: 18
Accuracy delta: 0.012088650100738785


In [108]:
# ============================================================
# 71. Selective trajectory-switch target
# ============================================================

switch_target = (
    (~sem_correct_oof)
    &
    trans_correct_oof
).astype(int)

print(
    "Total training examples:",
    len(switch_target),
)

print(
    "Switch opportunities:",
    switch_target.sum(),
)

print(
    "Do not switch:",
    (switch_target == 0).sum(),
)

print(
    "Switch rate:",
    switch_target.mean(),
)

Total training examples: 1489
Switch opportunities: 69
Do not switch: 1420
Switch rate: 0.04633982538616521


In [109]:
# ============================================================
# 72. Deployable switch features
# ============================================================

switch_X = np.hstack([
    # Predictions/confidence of both experts
    oof_sem_prob,
    oof_trans_prob,

    # Confidence / margin / entropy
    sem_meta,
    trans_meta,

    # How expert beliefs differ
    oof_trans_prob - oof_sem_prob,

    # Actual trajectory-transition features
    T_train,
])

print(
    "Switch feature matrix:",
    switch_X.shape,
)

assert switch_X.shape[0] == len(y_train_int)

Switch feature matrix: (1489, 40)


In [110]:
# ============================================================
# 73. Cross-fitted selective-switch model
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

switch_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

switch_oof_prob = np.zeros(
    len(y_train_int),
    dtype=np.float32,
)

for fold, (tr, va) in enumerate(
    switch_cv.split(
        switch_X,
        switch_target,
        groups=groups,
    ),
    start=1,
):

    model = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        ),
    )

    model.fit(
        switch_X[tr],
        switch_target[tr],
    )

    switch_oof_prob[va] = (
        model.predict_proba(
            switch_X[va]
        )[:, 1]
    )

    print(
        f"Fold {fold} complete"
    )

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [111]:
# ============================================================
# 74. Rescue-detection quality
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

print(
    "Positive prevalence:",
    switch_target.mean(),
)

print(
    "PR-AUC:",
    average_precision_score(
        switch_target,
        switch_oof_prob,
    ),
)

print(
    "ROC-AUC:",
    roc_auc_score(
        switch_target,
        switch_oof_prob,
    ),
)

Positive prevalence: 0.04633982538616521
PR-AUC: 0.22290062518658188
ROC-AUC: 0.893641559501939


In [112]:
# ============================================================
# 75. Selective intervention curve
# ============================================================

threshold_rows = []

for threshold in np.linspace(
    0.05,
    0.95,
    91,
):

    switch = (
        switch_oof_prob
        >= threshold
    )

    pred = (
        oof_sem_pred.copy()
    )

    pred[switch] = (
        oof_trans_pred[switch]
    )

    correct = (
        pred == y_train_int
    )

    rescues = np.sum(
        switch
        &
        (~sem_correct_oof)
        &
        trans_correct_oof
    )

    breaks = np.sum(
        switch
        &
        sem_correct_oof
        &
        (~trans_correct_oof)
    )

    threshold_rows.append({
        "threshold":
            threshold,

        "switches":
            int(switch.sum()),

        "coverage":
            switch.mean(),

        "rescues":
            int(rescues),

        "breaks":
            int(breaks),

        "net_rescues":
            int(rescues - breaks),

        "accuracy":
            correct.mean(),

        "macro_f1":
            f1_score(
                y_train_int,
                pred,
                average="macro",
                zero_division=0,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train_int,
                pred,
            ),
    })


intervention_curve = (
    pd.DataFrame(
        threshold_rows
    )
)

display(
    intervention_curve
    .sort_values(
        "net_rescues",
        ascending=False,
    )
    .head(15)
    .round(4)
)

,threshold,switches,coverage,rescues,breaks,net_rescues,accuracy,macro_f1,balanced_accuracy
62,0.67,231,0.1551,52,24,28,0.5440,0.5119,0.4967
60,0.65,242,0.1625,53,25,28,0.5440,0.5119,0.4967
63,0.68,227,0.1525,52,24,28,0.5440,0.5119,0.4967
61,0.66,237,0.1592,53,25,28,0.5440,0.5119,0.4967
57,0.62,267,0.1793,56,29,27,0.5433,0.5108,0.4953
59,0.64,254,0.1706,54,27,27,0.5433,0.5111,0.4958
56,0.61,273,0.1833,56,30,26,0.5426,0.5103,0.4950
66,0.71,205,0.1377,48,22,26,0.5426,0.5111,0.4964
65,0.70,210,0.1410,49,23,26,0.5426,0.5110,0.4961
64,0.69,217,0.1457,50,24,26,0.5426,0.5108,0.4957


In [113]:
# ============================================================
# 76. Fit final selective trajectory switch
# ============================================================

FINAL_SWITCH_C = 0.1
FINAL_THRESHOLD = 0.67


final_switch_model = make_pipeline(
    StandardScaler(),

    LogisticRegression(
        C=FINAL_SWITCH_C,
        class_weight="balanced",
        max_iter=5000,
        random_state=42,
    ),
)

final_switch_model.fit(
    switch_X,
    switch_target,
)

print(
    "✓ Final switch model fitted"
)

✓ Final switch model fitted


In [114]:
# ============================================================
# 77. Fit final experts
# ============================================================

final_semantic_model = make_pipeline(
    StandardScaler(),

    LogisticRegression(
        C=0.03,
        max_iter=5000,
        random_state=42,
    ),
)

final_transition_model = make_pipeline(
    StandardScaler(),

    LogisticRegression(
        C=0.01,
        max_iter=5000,
        random_state=42,
    ),
)


final_semantic_model.fit(
    X_sem_train,
    y_train_int,
)

final_transition_model.fit(
    X_trans_train,
    y_train_int,
)


test_sem_prob = (
    final_semantic_model
    .predict_proba(
        X_sem_test
    )
)

test_trans_prob = (
    final_transition_model
    .predict_proba(
        X_trans_test
    )
)


test_sem_pred = (
    test_sem_prob.argmax(axis=1)
)

test_trans_pred = (
    test_trans_prob.argmax(axis=1)
)

print(
    "✓ Final expert predictions generated"
)

✓ Final expert predictions generated


In [115]:
# ============================================================
# 78. Test router features
# ============================================================

test_sem_meta = probability_features(
    test_sem_prob
)

test_trans_meta = probability_features(
    test_trans_prob
)


test_switch_X = np.hstack([
    test_sem_prob,
    test_trans_prob,

    test_sem_meta,
    test_trans_meta,

    test_trans_prob
    - test_sem_prob,

    T_test,
])


print(
    "Train switch X:",
    switch_X.shape,
)

print(
    "Test switch X:",
    test_switch_X.shape,
)

assert (
    switch_X.shape[1]
    == test_switch_X.shape[1]
)

Train switch X: (1489, 40)
Test switch X: (287, 40)


In [116]:
# ============================================================
# 79. Final held-out selective intervention
# ============================================================

test_switch_prob = (
    final_switch_model
    .predict_proba(
        test_switch_X
    )[:, 1]
)


test_switch = (
    test_switch_prob
    >= FINAL_THRESHOLD
)


test_routed_pred = (
    test_sem_pred.copy()
)

test_routed_pred[
    test_switch
] = (
    test_trans_pred[
        test_switch
    ]
)


print(
    "Test examples:",
    len(y_test_int),
)

print(
    "Switched:",
    test_switch.sum(),
)

print(
    "Switch coverage:",
    test_switch.mean(),
)

Test examples: 287
Switched: 90
Switch coverage: 0.313588850174216


In [117]:
# ============================================================
# 80. FINAL HELD-OUT RESULT
# ============================================================

final_results = pd.DataFrame([
    {
        "model":
            "semantic",

        **evaluate_predictions(
            y_test_int,
            test_sem_pred,
        ),
    },

    {
        "model":
            "transition",

        **evaluate_predictions(
            y_test_int,
            test_trans_pred,
        ),
    },

    {
        "model":
            "selective_trajectory_correction",

        **evaluate_predictions(
            y_test_int,
            test_routed_pred,
        ),
    },
])


display(
    final_results.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic,0.4460,0.4702,0.4614,0.4582
1,transition,0.4843,0.4595,0.4661,0.4915
2,selective_trajectory_correction,0.4878,0.4662,0.4707,0.4957


In [118]:
# ============================================================
# 81. Final held-out rescue analysis
# ============================================================

sem_test_correct = (
    test_sem_pred
    == y_test_int
)

trans_test_correct = (
    test_trans_pred
    == y_test_int
)

routed_test_correct = (
    test_routed_pred
    == y_test_int
)


test_rescues = (
    (~sem_test_correct)
    &
    routed_test_correct
)

test_breaks = (
    sem_test_correct
    &
    (~routed_test_correct)
)


print(
    "Semantic correct:",
    sem_test_correct.sum(),
)

print(
    "Transition correct:",
    trans_test_correct.sum(),
)

print(
    "Routed correct:",
    routed_test_correct.sum(),
)

print(
    "\nSwitches:",
    test_switch.sum(),
)

print(
    "Rescues:",
    test_rescues.sum(),
)

print(
    "Breaks:",
    test_breaks.sum(),
)

print(
    "Net rescues:",
    test_rescues.sum()
    - test_breaks.sum(),
)

print(
    "\nAccuracy delta vs semantic:",
    routed_test_correct.mean()
    - sem_test_correct.mean(),
)

Semantic correct: 128
Transition correct: 139
Routed correct: 140

Switches: 90
Rescues: 28
Breaks: 16
Net rescues: 12

Accuracy delta vs semantic: 0.04181184668989546


In [119]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_int,
        test_routed_pred,
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0,
    )
)

                       precision    recall  f1-score   support

       workflow_error     0.6033    0.5290    0.5637       138
     constraint_error     0.4865    0.5143    0.5000        70
       tool_use_error     0.3514    0.3421    0.3467        38
grounding_state_error     0.2553    0.4000    0.3117        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.4878       287
            macro avg     0.4893    0.4662    0.4707       287
         weighted avg     0.5107    0.4878    0.4957       287



In [120]:
# ============================================================
# 82. Paired bootstrap confidence intervals
# ============================================================

import numpy as np

rng = np.random.default_rng(42)

N_BOOT = 10_000
n = len(y_test_int)

semantic_acc = []
transition_acc = []
routed_acc = []

route_vs_semantic = []
route_vs_transition = []

for _ in range(N_BOOT):

    idx = rng.integers(
        0,
        n,
        size=n,
    )

    y_b = y_test_int[idx]

    sem_b = test_sem_pred[idx]
    trans_b = test_trans_pred[idx]
    route_b = test_routed_pred[idx]

    sem_acc = np.mean(
        sem_b == y_b
    )

    trans_acc = np.mean(
        trans_b == y_b
    )

    route_acc = np.mean(
        route_b == y_b
    )

    semantic_acc.append(
        sem_acc
    )

    transition_acc.append(
        trans_acc
    )

    routed_acc.append(
        route_acc
    )

    route_vs_semantic.append(
        route_acc - sem_acc
    )

    route_vs_transition.append(
        route_acc - trans_acc
    )


def bootstrap_summary(values):

    values = np.asarray(values)

    return {
        "mean": values.mean(),
        "ci_low": np.quantile(
            values,
            0.025,
        ),
        "ci_high": np.quantile(
            values,
            0.975,
        ),
        "p_le_zero": np.mean(
            values <= 0
        ),
    }


print(
    "Router vs semantic:"
)

print(
    bootstrap_summary(
        route_vs_semantic
    )
)

print(
    "\nRouter vs transition:"
)

print(
    bootstrap_summary(
        route_vs_transition
    )
)

Router vs semantic:
{'mean': 0.041698257839721246, 'ci_low': -0.0034843205574912606, 'ci_high': 0.08710801393728224, 'p_le_zero': 0.0398}

Router vs transition:
{'mean': 0.0034613240418118465, 'ci_low': -0.013937282229965153, 'ci_high': 0.020905923344947785, 'p_le_zero': 0.4269}


In [121]:
# ============================================================
# 83. Exact McNemar test
# ============================================================

from scipy.stats import binomtest


def exact_mcnemar(
    pred_a,
    pred_b,
    y,
):

    a_correct = (
        pred_a == y
    )

    b_correct = (
        pred_b == y
    )

    # A wrong, B correct
    b_rescues = np.sum(
        (~a_correct)
        &
        b_correct
    )

    # A correct, B wrong
    b_breaks = np.sum(
        a_correct
        &
        (~b_correct)
    )

    discordant = (
        b_rescues
        + b_breaks
    )

    result = binomtest(
        b_rescues,
        n=discordant,
        p=0.5,
        alternative="two-sided",
    )

    return {
        "rescues": b_rescues,
        "breaks": b_breaks,
        "net": (
            b_rescues
            - b_breaks
        ),
        "discordant": discordant,
        "p_value": result.pvalue,
    }


print(
    "Router vs semantic"
)

print(
    exact_mcnemar(
        test_sem_pred,
        test_routed_pred,
        y_test_int,
    )
)


print(
    "\nRouter vs transition"
)

print(
    exact_mcnemar(
        test_trans_pred,
        test_routed_pred,
        y_test_int,
    )
)

Router vs semantic
{'rescues': 28, 'breaks': 16, 'net': 12, 'discordant': 44, 'p_value': 0.09614175442061423}

Router vs transition
{'rescues': 4, 'breaks': 3, 'net': 1, 'discordant': 7, 'p_value': 1.0}


In [122]:
# ============================================================
# 83. Exact McNemar test
# ============================================================

from scipy.stats import binomtest


def exact_mcnemar(
    pred_a,
    pred_b,
    y,
):

    a_correct = (
        pred_a == y
    )

    b_correct = (
        pred_b == y
    )

    # A wrong, B correct
    b_rescues = np.sum(
        (~a_correct)
        &
        b_correct
    )

    # A correct, B wrong
    b_breaks = np.sum(
        a_correct
        &
        (~b_correct)
    )

    discordant = (
        b_rescues
        + b_breaks
    )

    result = binomtest(
        b_rescues,
        n=discordant,
        p=0.5,
        alternative="two-sided",
    )

    return {
        "rescues": b_rescues,
        "breaks": b_breaks,
        "net": (
            b_rescues
            - b_breaks
        ),
        "discordant": discordant,
        "p_value": result.pvalue,
    }


print(
    "Router vs semantic"
)

print(
    exact_mcnemar(
        test_sem_pred,
        test_routed_pred,
        y_test_int,
    )
)


print(
    "\nRouter vs transition"
)

print(
    exact_mcnemar(
        test_trans_pred,
        test_routed_pred,
        y_test_int,
    )
)

Router vs semantic
{'rescues': 28, 'breaks': 16, 'net': 12, 'discordant': 44, 'p_value': 0.09614175442061423}

Router vs transition
{'rescues': 4, 'breaks': 3, 'net': 1, 'discordant': 7, 'p_value': 1.0}


# Summary — Local Transition Structure for Failure-Family Prediction

## Research Question

**Does recent trajectory history contain predictive information about an agent's failure family beyond the semantic representation of the current event, and if so, what form of historical information is useful?**

This notebook investigated whether trajectory context improves five-class failure-family prediction:

- `workflow_error`
- `constraint_error`
- `tool_use_error`
- `grounding_state_error`
- `reasoning_value_error`

The experiments used the canonical group-disjoint train/test split:

- **Train targets:** 1,489
- **Test targets:** 287
- **Train trajectory events:** 3,792
- **Test trajectory events:** 799

Historical-event reconstruction was verified against the exported trajectory dataset:

- Train exact history-count match: **1.0**
- Test exact history-count match: **1.0**

Therefore, all historical features used in this notebook were constructed from the intended pre-target trajectory events.

---

## 1. Current Semantic Baseline

A classifier using only the semantic embedding of the current target event provided a strong reference point.

Depending on the model-selection procedure, the semantic baseline achieved approximately:

- Accuracy: **0.446–0.516**
- Macro-F1: **0.461–0.490**

This established that the current event itself contains substantial information about the failure family.

The central question was therefore not whether history is predictive in isolation, but whether it provides **complementary information beyond current-event semantics**.

---

## 2. Direct Historical Semantic Aggregation

Several simple representations of historical semantic embeddings were evaluated:

- mean historical embedding
- recency-weighted historical embedding
- last-event embedding
- current + mean history
- current + recency history
- current + last event

History alone generally underperformed current semantics.

The strongest history-only representation was the immediately preceding event:

| Representation | Accuracy | Balanced Accuracy | Macro-F1 | Weighted-F1 |
|---|---:|---:|---:|---:|
| Current semantic | 0.5157 | 0.4814 | 0.4903 | 0.5215 |
| History mean | 0.5052 | 0.3782 | 0.3888 | 0.4895 |
| History recency | 0.4321 | 0.3540 | 0.3536 | 0.4260 |
| Last event | **0.5331** | 0.4477 | 0.4494 | 0.5112 |

Although the last event achieved higher raw accuracy than the current semantic representation, it had lower balanced accuracy and macro-F1.

This suggested that recent context contains signal, but simple historical aggregation does not consistently improve class-balanced prediction.

---

## 3. History Window Analysis

History windows of different lengths were evaluated.

For history-only representations, increasing the number of previous events did not produce consistent improvements. The immediately preceding event was strongest by raw accuracy.

When history was combined with current semantics, a short window of approximately three previous events performed best:

| History Window | Accuracy | Balanced Accuracy | Macro-F1 | Weighted-F1 |
|---:|---:|---:|---:|---:|
| 1 | 0.4983 | 0.4689 | 0.4720 | 0.5036 |
| 2 | 0.5017 | 0.4676 | 0.4718 | 0.5096 |
| 3 | **0.5122** | **0.4876** | **0.4888** | **0.5196** |
| 5 | 0.5017 | 0.4647 | 0.4718 | 0.5081 |
| 10 | 0.5122 | 0.4824 | 0.4857 | 0.5175 |

The last-three-event representation came very close to the current semantic baseline but did not materially exceed it in macro-F1.

The important result was that **long histories were not required**. Most useful historical information appeared to be local.

---

## 4. Text-Aware Sequential Modeling

A text-aware trajectory GRU was tested to determine whether a learned sequence encoder could extract useful temporal structure from the historical event embeddings.

The model achieved:

- Accuracy: **0.3554**
- Balanced accuracy: **0.3872**
- Macro-F1: **0.3819**
- Weighted-F1: **0.3606**

This substantially underperformed the simpler current-semantic and short-history representations.

Therefore, the failure of previous trajectory models was not simply caused by insufficient textual information in the historical events.

A more expressive sequence architecture did not automatically recover useful trajectory signal.

---

## 5. Relational History Features

The next hypothesis was that useful trajectory information may exist not in the historical embeddings themselves, but in the **relationship between the current event and recent events**.

Relational representations were therefore constructed for the previous 1–3 events.

The best relational window used the previous **two events**:

- Accuracy: **0.5122**
- Balanced accuracy: **0.4534**
- Macro-F1: **0.4678**
- Weighted-F1: **0.5187**

This did not outperform current semantics overall, but it motivated a more explicit representation of trajectory transitions.

---

## 6. Compact Transition Representation

A compact transition feature set was constructed from the current event and the previous three events.

Features included:

- cosine similarity between current and previous events
- L1 distance
- L2 distance
- dot-product similarity
- event-presence indicators
- cosine/L2 transitions between consecutive historical events

This produced a **19-dimensional explicit transition representation**.

Cross-validation selected:

- Transition model: **C = 0.01**
- Semantic model: **C = 0.03**

Held-out performance was:

| Model | Accuracy | Balanced Accuracy | Macro-F1 | Weighted-F1 |
|---|---:|---:|---:|---:|
| Semantic | 0.4460 | **0.4702** | 0.4614 | 0.4582 |
| Semantic + compact transition | **0.4843** | 0.4595 | **0.4661** | **0.4915** |

Relative to the CV-selected semantic classifier, transition information produced:

- Accuracy: **+0.0383**
- Balanced accuracy: **−0.0107**
- Macro-F1: **+0.0047**
- Weighted-F1: **+0.0333**

The improvement was therefore concentrated primarily in overall and support-weighted prediction rather than uniformly improving recall across classes.

---

## 7. Rescue/Break Analysis

Comparing semantic and transition predictions on the 287 held-out examples:

- Semantic correct: **128**
- Transition correct: **139**
- Both correct: **108**
- Both wrong: **128**
- Transition rescues: **31**
- Transition breaks: **20**
- Net rescues: **+11**

The models disagreed on **63 / 287 examples (21.95%)**.

This demonstrated that the transition representation was not merely reproducing semantic predictions.

Its largest benefit occurred for `workflow_error`:

- **24 rescues**
- **8 breaks**
- **+16 net corrections**

By contrast:

- `constraint_error`: +2 net
- `reasoning_value_error`: 0 net
- `grounding_state_error`: −2 net
- `tool_use_error`: −5 net

Thus, transition information was **class-dependent rather than universally beneficial**.

---

## 8. Transition Geometry and Rescues

Rescued examples showed substantially stronger semantic similarity between the current event and recent history.

For example, mean current-to-history cosine similarity for rescued examples was:

- `t-3`: **0.6125**
- `t-2`: **0.6601**
- `t-1`: **0.7273**

For examples where both models were wrong:

- `t-3`: 0.3734
- `t-2`: 0.4248
- `t-1`: 0.4411

For breaks:

- `t-3`: 0.3842
- `t-2`: 0.3502
- `t-1`: 0.4203

The same pattern appeared in historical transition similarities.

This provides evidence that transition features are particularly useful when the current failure event is **strongly semantically connected to the immediately preceding trajectory**.

The useful historical signal therefore appears to be structured and conditional rather than globally predictive.

---

## 9. Oracle Complementarity

Because the semantic and transition models made different errors, an oracle choosing the correct expert whenever either expert was correct would achieve:

- Semantic accuracy: **0.4460**
- Transition accuracy: **0.4843**
- Oracle accuracy: **0.5540**

Potential oracle gain over semantic:

**+10.80 percentage points**

This established meaningful complementarity between the two representations and motivated selective routing.

---

## 10. Learned Selective Routing

Out-of-fold predictions were generated to train a router without using held-out test labels.

OOF expert behavior:

- Semantic accuracy: **0.5252**
- Transition accuracy: **0.5353**
- Disagreement rate: **12.29%**
- Rescues: **69**
- Breaks: **54**
- Net transition rescues: **+15**
- Oracle accuracy: **0.5715**

The routing target was highly imbalanced:

- Positive prevalence: **4.63%**

Nevertheless, the router showed strong ranking ability:

- ROC-AUC: **0.8936**
- PR-AUC: **0.2229**

Threshold analysis indicated that selective switching was preferable to unrestricted replacement of the semantic expert.

The selected held-out router produced:

- Semantic correct: **128**
- Transition correct: **139**
- Routed correct: **140**
- Switches: **90**
- Router rescues vs semantic: **28**
- Router breaks vs semantic: **16**
- Net rescues: **+12**

Final routed performance:

- Accuracy: **0.4878**
- Balanced accuracy: **0.4662**
- Macro-F1: **0.4707**
- Weighted-F1: **0.4957**

Relative to the CV-selected semantic baseline, accuracy increased by approximately:

**+4.18 percentage points**

---

## 11. Statistical Validation

The routed model was evaluated using paired bootstrap analysis and exact McNemar tests.

### Router vs Semantic

Observed accuracy improvement:

**+0.0418**

Bootstrap estimate:

- Mean difference: **+0.0417**
- 95% CI: **[-0.0035, 0.0871]**

Paired outcomes:

- Rescues: **28**
- Breaks: **16**
- Net: **+12**
- Discordant predictions: **44**

Exact McNemar test:

**p = 0.0961**

The observed improvement is therefore directionally positive but **not statistically significant at α = 0.05**.

### Router vs Transition

Observed improvement:

**+0.0035**

95% bootstrap CI:

**[-0.0139, 0.0209]**

Paired outcomes:

- Rescues: 4
- Breaks: 3
- Net: +1

Exact McNemar test:

**p = 1.0**

There is therefore no evidence that the router materially improves over the transition expert itself.

---

# Notebook Conclusion

The experiments show that trajectory history contains predictive information about failure families, but **the useful signal is not well represented by full-history aggregation or generic sequence encoders**.

Instead, the strongest evidence points toward **local semantic transition structure**.

Recent events—particularly the previous 1–3 events—contain most of the useful historical information. Explicit similarities and distances between the current event and recent history reveal complementary information that is lost when trajectory embeddings are simply pooled or passed through a generic recurrent encoder.

The compact transition representation improved held-out accuracy relative to the CV-selected semantic model and produced 31 semantic-error rescues against 20 breaks.

Selective routing exploited part of this complementarity, producing an observed +4.18 percentage-point accuracy improvement over the semantic expert. However, this improvement was not statistically significant under the exact paired test, and routing did not significantly outperform the transition expert.

The evidence therefore supports **local transition modeling** as the main contribution of this notebook, while selective routing should be regarded as promising but not yet statistically established.